# Six-month cash forecast

Built one step at a time. So far:

1. **Scope** - which countries, and which months, set in the notebook.
2. **Invoice dates** - each month spread across that country's work days.
3. **Payment terms** - each invoiced day split across the terms, in nominal days.
4. **Duty** - budgeted in its own right on each side, not a rate applied to revenue.
5. **Four streams** - the same decomposition run for the service and for the duty
   riding on it, on the money-in side (customer invoices) and the money-out side
   (COGS invoices from providers).
6. **Delays** - each due date shifted by how late that country and term actually pays.
7. **Payment dates** - the result rolled forward onto a work day.
8. **Factoring** - the countries that sell invoices to a bank settle them on a
   weekly clock of their own, in four legs rather than one.
9. **Cost timing** - manpower and SG&A reach cash on a rule each rather than on a
   term, at week grain.

No amounts are needed here. Every figure is a share of its stream's monthly budget, and
the amounts are applied in the workbook exported at the end.

The shares are **within a month and within a stream**: the four streams are shares of
four different budgets, so none of them can be netted against another until the amounts
are applied.

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

### Settings

Everything worth changing lives in the next cell, so you can set the model up without
reading the rest. The sections below explain what each of these means and what the code
around them does with it, but none of them are set anywhere else.

Two things are not in there because they are code rather than numbers. The bank holiday
rules in section 3 are functions, one per country, since a holiday like Easter Monday has
to be worked out rather than typed. And the P&L layout in the Excel section is the shape
of the sheet itself, not a setting.


In [ ]:
# ======================================================================================
# SETTINGS
#
# Everything the model runs on. Nothing below this cell sets any of it, so this is the
# only place to change the forecast. Two exceptions, both further down and both code
# rather than numbers: the bank holiday rules in section 3, and the P&L layout in the
# Excel section.
# ======================================================================================

# --- Scope: who is in the forecast, and over what months -----------------------------------
COUNTRIES = ["Denmark", "Norway", "Sweden", "United States"]
START = "2026-01"      # first month of the horizon
HORIZON_MONTHS = 24

# What to show instead of the full name in visuals and exports.
COUNTRY_LABELS = {
    "Denmark": "DNK",
    "Norway": "NOR",
    "Sweden": "SWE",
    "United States": "USA",
}

# --- Output: the file that gets written, and what the streams are called in it -------------
WORKBOOK = "forecast_6m_final_20260816_B.xlsx"

# What the streams are called on the way out, the same way COUNTRY_LABELS does for
# countries. Insertion order is service-then-duty per side, which is the order the
# stacked charts and the budget dropdown both read.
STREAM_LABELS = {"revenue": "Revenue", "revenue_duty": "Revenue duty",
                 "cogs": "COGS", "cogs_duty": "COGS duty"}

# --- Payment terms: how long a customer or a provider has to pay ---------------------------
PAYMENT_TERMS = {
    "Cash":                   ("invoice", 0),
    "Net +7 days":            ("invoice", 7),
    "Net +14 days":           ("invoice", 14),
    "Net +21 days":           ("invoice", 21),
    "Net +30 days":           ("invoice", 30),
    "Net +45 days":           ("invoice", 45),
    "Net +60 days":           ("invoice", 60),
    "Current Month +14 days": ("month_end", 14),
    "Current Month +21 days": ("month_end", 21),
    "Current Month +30 days": ("month_end", 30),
    "Current Month +45 days": ("month_end", 45),
    "Current Month +60 days": ("month_end", 60),
}

# --- The customer mix: how our revenue is split across those terms -------------------------
# Share of revenue invoicing on each term, per country. One column per country, so
# a new country is a new column. Placeholder weights - replace with the real mix.
# Each column is normalised later, so it does not have to add up to 100.
CUSTOMER_MIX = pd.DataFrame.from_dict(
    {                          # Denmark  Norway  Sweden  United States
        "Cash":                   (   5,      5,      5,      5),
        "Net +7 days":            (   5,      5,      5,      0),
        "Net +14 days":           (  20,     20,     15,      5),
        "Net +21 days":           (   5,      5,      5,      5),
        "Net +30 days":           (  40,     40,     40,     45),
        "Net +45 days":           (   5,      5,     10,     15),
        "Net +60 days":           (   5,      5,      5,     15),
        "Current Month +14 days": (   5,      5,      5,      0),
        "Current Month +21 days": (   0,      0,      0,      0),
        "Current Month +30 days": (  10,     10,     10,     10),
        "Current Month +45 days": (   0,      0,      0,      0),
        "Current Month +60 days": (   0,      0,      0,      0),
    },
    orient="index", columns=["Denmark", "Norway", "Sweden", "United States"],
)

# --- The provider mix: who we buy COGS from, and on what term ------------------------------
# Who invoices us for COGS, and on what term. Global: a provider's terms are its own.
PROVIDERS = {
    "Provider 1": "Net +45 days",
    "Provider 2": "Net +60 days",
}

# How COGS spend is split between them, per country. Same 80/20 everywhere for now -
# change a column and only that country moves.
PROVIDER_MIX = pd.DataFrame.from_dict(
    {               # Denmark  Norway  Sweden  United States
        "Provider 1": (  80,     80,     80,     80),
        "Provider 2": (  20,     20,     20,     20),
    },
    orient="index", columns=["Denmark", "Norway", "Sweden", "United States"],
)

# --- The four traded streams, and which side of the cash flow each lands on ----------------
# The four streams, and which side of the cash flow each lands on. Everything that has
# to tell money in from money out reads these rather than testing for a stream by name.
STREAMS_IN = ("revenue", "revenue_duty")
STREAMS_OUT = ("cogs", "cogs_duty")

# Which duty stream belongs to which service stream. The pair shares a term mix and a
# delay table; all that separates them is the budget.
DUTY_OF = {"revenue": "revenue_duty", "cogs": "cogs_duty"}

# --- Payment delays: how late each country and term actually pays --------------------------
# Average delay in calendar days between due date and payment, per country and term.
# Placeholder statistics - replace with your own. 0 means paid on the due date.
DELAY_CUSTOMER = pd.DataFrame.from_dict(
    {                          # Denmark  Norway  Sweden  United States
        "Cash":                   (   0,      0,      0,      0),
        "Net +7 days":            (   2,      2,      2,      4),
        "Net +14 days":           (   3,      3,      3,      6),
        "Net +21 days":           (   3,      3,      4,      7),
        "Net +30 days":           (   4,      4,      5,      9),
        "Net +45 days":           (   5,      5,      6,     11),
        "Net +60 days":           (   6,      6,      7,     13),
        "Current Month +14 days": (   3,      3,      4,      6),
        "Current Month +21 days": (   3,      3,      4,      7),
        "Current Month +30 days": (   4,      4,      5,      9),
        "Current Month +45 days": (   5,      5,      6,     11),
        "Current Month +60 days": (   6,      6,      7,     13),
    },
    orient="index", columns=["Denmark", "Norway", "Sweden", "United States"],
)

# We pay providers on the due date for now - no delay assumed. Spell this out as a
# full table like the one above once there are real statistics per provider.
DELAY_PROVIDER = pd.DataFrame(0, index=list(PAYMENT_TERMS), columns=COUNTRIES)

# Duty runs as late as whatever it is invoiced alongside, so each duty stream shares its
# partner's table rather than getting one of its own.
DELAYS = {
    "revenue":      DELAY_CUSTOMER,
    "revenue_duty": DELAY_CUSTOMER,
    "cogs":         DELAY_PROVIDER,
    "cogs_duty":    DELAY_PROVIDER,
}

# --- Factoring: which countries sell invoices to a bank, in which months, and on what terms
# All four of these are per country *and* per month. A facility is negotiated and
# renegotiated rather than settled once: a country can take one on halfway through the
# horizon, give it up again, or keep it and sell less through it, and the advance and the
# fee move with the same conversation. So each of them is a table of one row per month and
# one column per country, and a single month in a single country is a single cell.
#
# The rows have to cover START through HORIZON_MONTHS exactly. Section 10 raises where one
# is missing, rather than reading the nearest month it can find and quietly forecasting
# something nobody asked for.
#
# Every cell is seeded with what its country carried before any of this was per month, so
# the model is exactly what it was until one of them is edited.

# Does this country sell part of its receivables to a bank in this month at all? A month
# with no facility runs exactly as it did before this section existed: one leg, worth all
# of it. Whether the country carries the other three legs at all is still decided once for
# the whole horizon - see factors_ever() in section 10 - because the rows are written when
# this notebook runs and it is the rate on them that empties a month answering No.
FACTORING = pd.DataFrame.from_dict(
    {             # Denmark  Norway  Sweden  United States
        "2026-01": (   True,  False,  False,         False),
        "2026-02": (   True,  False,  False,         False),
        "2026-03": (   True,  False,  False,         False),
        "2026-04": (   True,  False,  False,         False),
        "2026-05": (   True,  False,  False,         False),
        "2026-06": (   True,  False,  False,         False),
        "2026-07": (   True,  False,  False,         False),
        "2026-08": (   True,  False,  False,         False),
        "2026-09": (   True,  False,  False,         False),
        "2026-10": (   True,  False,  False,         False),
        "2026-11": (   True,  False,  False,         False),
        "2026-12": (   True,  False,  False,         False),
        "2027-01": (   True,  False,  False,         False),
        "2027-02": (   True,  False,  False,         False),
        "2027-03": (   True,  False,  False,         False),
        "2027-04": (   True,  False,  False,         False),
        "2027-05": (   True,  False,  False,         False),
        "2027-06": (   True,  False,  False,         False),
        "2027-07": (   True,  False,  False,         False),
        "2027-08": (   True,  False,  False,         False),
        "2027-09": (   True,  False,  False,         False),
        "2027-10": (   True,  False,  False,         False),
        "2027-11": (   True,  False,  False,         False),
        "2027-12": (   True,  False,  False,         False),
    },
    orient="index", columns=["Denmark", "Norway", "Sweden", "United States"],
)

# How much of that country's invoicing the bank buys in this month, what it pays out
# against it, and what it keeps. Placeholder figures - replace them with what the facility
# actually says, month by month where it says something different.
#
# The fee is a share of the invoice *face value*, not of the retention it is taken out of.
FACTORED_SHARE = pd.DataFrame.from_dict(
    {             # Denmark  Norway  Sweden  United States
        "2026-01": (   0.40,   0.55,   0.00,          0.00),
        "2026-02": (   0.40,   0.55,   0.00,          0.00),
        "2026-03": (   0.40,   0.55,   0.00,          0.00),
        "2026-04": (   0.40,   0.55,   0.00,          0.00),
        "2026-05": (   0.40,   0.55,   0.00,          0.00),
        "2026-06": (   0.40,   0.55,   0.00,          0.00),
        "2026-07": (   0.40,   0.55,   0.00,          0.00),
        "2026-08": (   0.40,   0.55,   0.00,          0.00),
        "2026-09": (   0.40,   0.55,   0.00,          0.00),
        "2026-10": (   0.40,   0.55,   0.00,          0.00),
        "2026-11": (   0.40,   0.55,   0.00,          0.00),
        "2026-12": (   0.40,   0.55,   0.00,          0.00),
        "2027-01": (   0.40,   0.55,   0.00,          0.00),
        "2027-02": (   0.40,   0.55,   0.00,          0.00),
        "2027-03": (   0.40,   0.55,   0.00,          0.00),
        "2027-04": (   0.40,   0.55,   0.00,          0.00),
        "2027-05": (   0.40,   0.55,   0.00,          0.00),
        "2027-06": (   0.40,   0.55,   0.00,          0.00),
        "2027-07": (   0.40,   0.55,   0.00,          0.00),
        "2027-08": (   0.40,   0.55,   0.00,          0.00),
        "2027-09": (   0.40,   0.55,   0.00,          0.00),
        "2027-10": (   0.40,   0.55,   0.00,          0.00),
        "2027-11": (   0.40,   0.55,   0.00,          0.00),
        "2027-12": (   0.40,   0.55,   0.00,          0.00),
    },
    orient="index", columns=["Denmark", "Norway", "Sweden", "United States"],
)

ADVANCE_RATE = pd.DataFrame.from_dict(
    {             # Denmark  Norway  Sweden  United States
        "2026-01": (   0.90,   0.90,   0.90,          0.90),
        "2026-02": (   0.90,   0.90,   0.90,          0.90),
        "2026-03": (   0.90,   0.90,   0.90,          0.90),
        "2026-04": (   0.90,   0.90,   0.90,          0.90),
        "2026-05": (   0.90,   0.90,   0.90,          0.90),
        "2026-06": (   0.90,   0.90,   0.90,          0.90),
        "2026-07": (   0.90,   0.90,   0.90,          0.90),
        "2026-08": (   0.90,   0.90,   0.90,          0.90),
        "2026-09": (   0.90,   0.90,   0.90,          0.90),
        "2026-10": (   0.90,   0.90,   0.90,          0.90),
        "2026-11": (   0.90,   0.90,   0.90,          0.90),
        "2026-12": (   0.90,   0.90,   0.90,          0.90),
        "2027-01": (   0.90,   0.90,   0.90,          0.90),
        "2027-02": (   0.90,   0.90,   0.90,          0.90),
        "2027-03": (   0.90,   0.90,   0.90,          0.90),
        "2027-04": (   0.90,   0.90,   0.90,          0.90),
        "2027-05": (   0.90,   0.90,   0.90,          0.90),
        "2027-06": (   0.90,   0.90,   0.90,          0.90),
        "2027-07": (   0.90,   0.90,   0.90,          0.90),
        "2027-08": (   0.90,   0.90,   0.90,          0.90),
        "2027-09": (   0.90,   0.90,   0.90,          0.90),
        "2027-10": (   0.90,   0.90,   0.90,          0.90),
        "2027-11": (   0.90,   0.90,   0.90,          0.90),
        "2027-12": (   0.90,   0.90,   0.90,          0.90),
    },
    orient="index", columns=["Denmark", "Norway", "Sweden", "United States"],
)

FACTORING_FEE = pd.DataFrame.from_dict(
    {             # Denmark  Norway  Sweden  United States
        "2026-01": (   0.01,   0.01,   0.01,          0.01),
        "2026-02": (   0.01,   0.01,   0.01,          0.01),
        "2026-03": (   0.01,   0.01,   0.01,          0.01),
        "2026-04": (   0.01,   0.01,   0.01,          0.01),
        "2026-05": (   0.01,   0.01,   0.01,          0.01),
        "2026-06": (   0.01,   0.01,   0.01,          0.01),
        "2026-07": (   0.01,   0.01,   0.01,          0.01),
        "2026-08": (   0.01,   0.01,   0.01,          0.01),
        "2026-09": (   0.01,   0.01,   0.01,          0.01),
        "2026-10": (   0.01,   0.01,   0.01,          0.01),
        "2026-11": (   0.01,   0.01,   0.01,          0.01),
        "2026-12": (   0.01,   0.01,   0.01,          0.01),
        "2027-01": (   0.01,   0.01,   0.01,          0.01),
        "2027-02": (   0.01,   0.01,   0.01,          0.01),
        "2027-03": (   0.01,   0.01,   0.01,          0.01),
        "2027-04": (   0.01,   0.01,   0.01,          0.01),
        "2027-05": (   0.01,   0.01,   0.01,          0.01),
        "2027-06": (   0.01,   0.01,   0.01,          0.01),
        "2027-07": (   0.01,   0.01,   0.01,          0.01),
        "2027-08": (   0.01,   0.01,   0.01,          0.01),
        "2027-09": (   0.01,   0.01,   0.01,          0.01),
        "2027-10": (   0.01,   0.01,   0.01,          0.01),
        "2027-11": (   0.01,   0.01,   0.01,          0.01),
        "2027-12": (   0.01,   0.01,   0.01,          0.01),
    },
    orient="index", columns=["Denmark", "Norway", "Sweden", "United States"],
)

# --- Factoring: what the bank buys, and when it pays ---------------------------------------
# The bank buys the invoice, and duty is a line on that invoice, so it is sold with it.
FACTORED_STREAMS = ("revenue", "revenue_duty")

# The four ways money reaches us, and what they are called on the way out - the same way
# STREAM_LABELS does for streams. Every row carries one.
LEG_LABELS = {
    "direct":    "Direct",               # the customer pays us, on the terms in section 5
    "advance":   "Factoring advance",    # the bank pays out, on its own weekly clock
    "retention": "Factoring retention",  # the rest, once the customer has paid the bank
    "fee":       "Factoring fee",        # what the bank keeps out of it, and money out
}

INVOICE_RUN_WEEKDAY = 0    # Monday - the day the week's factored invoices go out
ADVANCE_OFFSET_DAYS = 3    # Thursday - three days on, when the bank settles that run
BANK_DELAY_DAYS = 1        # the retention reaches us a day behind the customer's payment

# --- Factoring: where the facility switch lives --------------------------------------------
# Whether the facility itself is a notebook setting. Option B: it is not, so every country
# carries the factoring legs whether it uses them or not and the rate on the workbook is
# what empties them. Turning one on is typing Yes on the Factoring sheet - at the cost of
# the rows a country that never factors still drags around.
FACTORING_IS_A_NOTEBOOK_SETTING = False

# --- Cost timing: which rule each cost line pays on, per country ---------------------------
# The cost lines that move on a schedule of their own rather than on a payment term. The
# key is the P&L row each one is budgeted on, so a figure typed on a P&L sheet drives its
# cash the same way revenue and COGS already do.
SCHEDULED_LINES = {
    "mp_courier":   "Manpower courier",
    "mp_ff":        "Manpower freight forward",
    "sga_staff":    "Staff cost",
    "sga_vehicle":  "Vehicle",
    "sga_rent":     "Rent",
    "sga_it":       "IT",
    "sga_sales":    "Sales & Marketing",
    "sga_travel":   "Travel & Rep",
    "sga_admin":    "Admin & Misc",
    "sga_bad_debt": "Bad debt",
    "sga_sla":      "SLA",
    "sga_external": "External Assistance",
}

TIMING_RULES = {
    "month_end": "Last working day of month",
    "mid_month": "Mid month, the 15th",
    "spread":    "Equal spread over working days",
    "week":      "One week of the month",
    "none":      "No cash effect",
}

COST_TIMING = pd.DataFrame.from_dict(
    {                    # Denmark      Norway       Sweden       United States
        "mp_courier":   ("month_end", "month_end", "month_end", "month_end"),
        "mp_ff":        ("month_end", "month_end", "month_end", "month_end"),
        "sga_staff":    ("month_end", "month_end", "month_end", "month_end"),
        "sga_vehicle":  ("mid_month", "mid_month", "mid_month", "mid_month"),
        "sga_rent":     ("month_end", "month_end", "month_end", "month_end"),
        "sga_it":       ("spread",    "spread",    "spread",    "spread"),
        "sga_sales":    ("spread",    "spread",    "spread",    "spread"),
        "sga_travel":   ("spread",    "spread",    "spread",    "spread"),
        "sga_admin":    ("spread",    "spread",    "spread",    "spread"),
        # Written off rather than paid out, so it never reaches cash.
        "sga_bad_debt": ("none",      "none",      "none",      "none"),
        "sga_sla":      ("month_end", "month_end", "month_end", "month_end"),
        "sga_external": ("month_end", "month_end", "month_end", "month_end"),
    },
    orient="index", columns=["Denmark", "Norway", "Sweden", "United States"],
)

# --- Cost timing: the two parameters the rules take ----------------------------------------
# Which week the "one week of the month" rule means. Nothing uses it yet - it is here so a
# cost that really does land in, say, the second week has somewhere to go without this
# section having to be rewritten for it. Clamped to the weeks a month actually has.
COST_TIMING_WEEK = pd.DataFrame(2, index=list(SCHEDULED_LINES), columns=COUNTRIES)

MID_MONTH_DAY = 15

# --- The budget: what is closed, what unit the P&L is in, and how big each country is ------
FIRST_BUDGET_MONTH = 5             # the first year is closed to April; May onwards is budget

# The sheet is kept in thousands - DKKt - and every amount elsewhere in the workbook is in
# whole units. This is the factor between them, applied where the budget sheet reads a
# figure off a P&L.
PL_UNIT_SCALE = 1000

# The unit cell, which doubles as the currency. Every P&L is kept in DKK thousands
# whichever country it is for, so the workbook has one reporting currency throughout and
# amounts are addable across countries. What it does not hold is the rate each country's
# budget was translated at: that happens before a figure is typed on a sheet.
PL_UNIT = "DKKt"

# --- Charts: the palette everything is drawn in --------------------------------------------
# Categorical colours in a fixed order, so nothing gets repainted when the list
# changes. Slot 4 is violet rather than yellow: yellow sits too close to slot 2's
# orange to be told apart when both are on screen.
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#4a3aa7", "#eda100", "#e87ba4", "#008300", "#e34948"]
INK, MUTED, GRID, RULE = "#0b0b0b", "#898781", "#e1e0d9", "#c3c2b7"

# --- Charts: how big the ones in the workbook are ------------------------------------------
# Long and low: the axis carries a hundred-odd weeks over a two-year horizon, and
# squeezing those into a square leaves the bars too narrow to read.
CHART_WIDTH, CHART_HEIGHT = 34, 8  # centimetres
CHART_ROWS = round(CHART_HEIGHT / 0.53) + 1   # default rows one chart covers, for stacking
CHART_GAP = 1                      # blank columns between the table and the blocks beside it
PANEL_GAP = 3                      # blank rows between one panel's block and the next

LABEL_ROTATION = -5400000   # -90 degrees, in the 1/60000ths of a degree OOXML counts in
TARGET_LABELS = 20          # roughly how many labels a chart carries, whatever its length
LABEL_FLOOR = 0.06          # no label on a bar under this share of the tallest in its panel


## 2. Scope

Which countries and which months to build the forecast for. Edit these directly -
adding a country means giving it a calendar, a term mix, a provider mix, a duty
rate and a display label, each of which raises if you forget.

Countries are spelled out in full in the data. `COUNTRY_LABELS` holds the short
code to show instead in charts and exports, so the display never has to leak back
into the keys.

In [ ]:
def require_all_countries(config, what):
    """Every country needs its own entry - a missing one would quietly skew the forecast."""
    missing = [country for country in COUNTRIES if country not in config]
    if missing:
        raise KeyError(f"No {what} defined for: {missing}")


require_all_countries(COUNTRY_LABELS, "display label")


def label(country):
    """Short display code for one country. Display only - never used as a key."""
    return COUNTRY_LABELS[country]


def with_labels(df, column="country"):
    """A copy with the country column swapped for its display code, ready to export."""
    return df.assign(**{column: df[column].map(COUNTRY_LABELS)})


# Every country x month in the horizon. freq="MS" gives the first day of each month.
periods = pd.date_range(START, periods=HORIZON_MONTHS, freq="MS")
scope = pd.DataFrame([(country, period) for country in COUNTRIES for period in periods],
                     columns=["country", "period"])
scope.head()

## 3. Invoice dates: spread each month across its work days

Each month is spread **evenly** over its work days, so a work day carries
`1 / number of work days` of that month.

Equal split is a placeholder. The real spread will be uneven - invoicing clusters
at certain points in the month - and `distribute()` is the single function to
change when we get to it.

Work days are Mon-Fri minus that country's bank holidays, which is why this is
done per country: the same month is worth a different daily share in each one.

The holidays are **rules, not dates** - Easter and the days hung off it, the fixed
national days, and the American observance rule that moves a Saturday holiday to the
Friday. They are generated for every year the horizon touches, so the calendars cannot
fall behind it the way a typed list of dates does.

In [ ]:
from dateutil.easter import easter
from pandas.tseries.holiday import USFederalHolidayCalendar

# Bank holidays as rules rather than as dates. A typed list covers the years somebody
# typed and quietly stops: the horizon ran into 2027 with a 2026-only calendar, so every
# 2027 month lost its weekends and nothing else, and the work-day counts came out high.
# Rules cannot fall behind the horizon - move START or HORIZON_MONTHS and the calendars
# follow. Still best-effort national calendars; check them against your own source before
# this drives anything real.
FRIDAY, SATURDAY = 4, 5


def fixed(year, month, day):
    """A holiday on the same date every year."""
    return pd.Timestamp(year=year, month=month, day=day)


def from_easter(year, days):
    """A holiday counted from Easter Sunday, which is most of the Nordic calendar."""
    return pd.Timestamp(easter(year)) + pd.Timedelta(days=days)


def weekday_from(date, weekday):
    """The first such weekday on or after a date - Sweden hangs two holidays off these."""
    return date + pd.Timedelta(days=(weekday - date.weekday()) % 7)


def danish_holidays(year):
    """Store bededag is deliberately absent: abolished as a public holiday from 2024."""
    return [fixed(year, 1, 1),        # Nytarsdag
            from_easter(year, -3),    # Skaertorsdag
            from_easter(year, -2),    # Langfredag
            from_easter(year, 1),     # 2. paskedag
            from_easter(year, 39),    # Kristi himmelfartsdag
            from_easter(year, 50),    # 2. pinsedag
            fixed(year, 6, 5),        # Grundlovsdag
            fixed(year, 12, 25), fixed(year, 12, 26)]


def norwegian_holidays(year):
    """17 May can fall on 2. pinsedag - 2027 does - so the list is deduplicated below."""
    return [fixed(year, 1, 1),
            from_easter(year, -3),    # Skjaertorsdag
            from_easter(year, -2),    # Langfredag
            from_easter(year, 1),     # 2. paskedag
            fixed(year, 5, 1),        # Arbeidernes dag
            fixed(year, 5, 17),       # Grunnlovsdagen
            from_easter(year, 39),    # Kristi himmelfartsdag
            from_easter(year, 50),    # 2. pinsedag
            fixed(year, 12, 25), fixed(year, 12, 26)]


def swedish_holidays(year):
    """No Maundy Thursday and no Whit Monday - the latter went in 2005, for 6 June.

    Midsummer's Eve and Christmas Eve are not red days and nobody works them, so they are
    in here on the same footing as the ones that are.
    """
    return [fixed(year, 1, 1),
            fixed(year, 1, 6),        # Trettondedag jul
            from_easter(year, -2),    # Langfredag
            from_easter(year, 1),     # Annandag pask
            fixed(year, 5, 1),        # Forsta maj
            from_easter(year, 39),    # Kristi himmelsfardsdag
            fixed(year, 6, 6),        # Nationaldagen
            weekday_from(fixed(year, 6, 19), FRIDAY),      # Midsommarafton
            weekday_from(fixed(year, 10, 31), SATURDAY),   # Alla helgons dag
            fixed(year, 12, 24), fixed(year, 12, 25), fixed(year, 12, 26)]


def american_holidays(year):
    """The federal calendar out of pandas, which already knows the observance rule.

    A federal holiday on a Saturday is kept the Friday before and one on a Sunday the
    Monday after, so 4 July 2026 is a Saturday and comes back as the 3rd. That rule is
    also why a year can carry the next year's New Year: 1 January 2028 is a Saturday, so
    it lands on 31 December 2027.
    """
    return list(USFederalHolidayCalendar().holidays(f"{year}-01-01", f"{year}-12-31"))


HOLIDAY_RULES = {
    "Denmark": danish_holidays,
    "Norway": norwegian_holidays,
    "Sweden": swedish_holidays,
    "United States": american_holidays,
}
require_all_countries(HOLIDAY_RULES, "bank holiday rules")

# Every year the horizon touches, so the calendars cover it by construction.
HOLIDAY_YEARS = range(periods.min().year, periods.max().year + 1)
HOLIDAYS = {country: sorted({date for year in HOLIDAY_YEARS for date in rule(year)})
            for country, rule in HOLIDAY_RULES.items()}

# One work-day calendar per country: Mon-Fri minus that country's bank holidays.
# Everything that needs to know what a work day is goes through these.
CALENDARS = {country: pd.offsets.CustomBusinessDay(holidays=HOLIDAYS[country])
             for country in COUNTRIES}


def work_days(country, period):
    """The work days inside one month."""
    return pd.bdate_range(period, period + pd.offsets.MonthEnd(1), freq=CALENDARS[country])


def distribute(scope):
    """One row per country / work day, carrying that day's share of its month."""
    rows = []
    for row in scope.itertuples():
        days = work_days(row.country, row.period)
        rows.append(pd.DataFrame({
            "country": row.country,
            "period": row.period,
            "invoice_date": days,
            # Equal split for now - every work day takes the same share.
            "pct_of_month": 1 / len(days),
        }))
    return pd.concat(rows, ignore_index=True).sort_values(["country", "invoice_date"])


invoiced = distribute(scope)
invoiced.head()

In [ ]:
# Every country-month must add back up to 100%.
invoiced.groupby(["country", "period"])["pct_of_month"].sum().describe()[["min", "max"]]

## 4. Share of the month per work day

One panel per country, on a shared scale. Each line is flat within a month and
steps at month boundaries: fewer work days in a month means each of them carries
a bigger share.

Countries get their own panel rather than sharing one, because their lines sit on
top of each other in every month where the holidays happen to match, and whichever
is drawn last hides the other.

April and May are where the four separate: the Nordics lose Easter and Ascension
days that the US does not, while the US loses days in months the Nordics work
straight through.

In [ ]:
def tint(colour, strength=0.45):
    """A washed-out version of a palette colour, for the duty half of a stream pair.

    Same hue, lighter: duty reads as part of the stream it sits on rather than as a
    fifth unrelated thing, which a new colour from the palette would.
    """
    channels = (int(colour[position:position + 2], 16) for position in (1, 3, 5))
    return "#" + "".join(f"{round(c + (255 - c) * strength):02x}" for c in channels)


def style(ax, title, colour):
    """Shared panel styling: recessive grid and axes, the panel's name as its title."""
    ax.set_title(title, loc="left", fontsize=10, color=colour, fontweight="bold", pad=4)
    ax.yaxis.set_major_formatter(lambda v, _: f"{abs(v):.1f}%")  # abs: cash out is drawn below zero
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(RULE)
    ax.tick_params(colors=MUTED)


by_country = list(invoiced.groupby("country"))
fig, axes = plt.subplots(len(by_country), 1, figsize=(12, 2 * len(by_country)),
                         sharex=True, sharey=True, constrained_layout=True)

for ax, colour, (country, rows) in zip(axes, PALETTE, by_country):
    ax.plot(rows["invoice_date"], rows["pct_of_month"] * 100, color=colour, linewidth=2)
    style(ax, label(country), colour)

fig.suptitle("Share of the month invoiced on each work day", fontsize=13, x=0.01, ha="left")
fig.supylabel("% of the month", fontsize=10, color=MUTED)
plt.show()

## 5. Payment terms

A term is `(anchor, days)`:

- `invoice` - counted from the invoice date itself (`Net +30 days`).
- `month_end` - counted from the last day of the invoice month
  (`Current Month +30 days`), so every invoice in a month falls due on the
  same day.

Due dates are **nominal** days, not work days: a due date can and does land on a
weekend or a bank holiday. Only invoicing is restricted to work days.

`PAYMENT_TERMS` is global - `Net +30 days` means the same arithmetic everywhere.
What differs is the **mix**: the share of invoicing sitting on each term. There is
one mix per stream, and each is a table with one column per country.

To add a term, add a line to `PAYMENT_TERMS` and a row to the mixes. A new kind of
anchor needs one extra branch in `due_date()`.

In [ ]:
def due_date(invoice_date, term):
    """When one invoice falls due. Nominal days: this can land on a weekend or holiday."""
    anchor, days = PAYMENT_TERMS[term]
    # MonthEnd(0) leaves a date that is already month end alone, and rolls the rest forward.
    start = invoice_date if anchor == "invoice" else invoice_date + pd.offsets.MonthEnd(0)
    return start + pd.Timedelta(days=days)

### The customer mix - how our customers pay us

In [ ]:
require_all_countries(CUSTOMER_MIX.columns, "customer term mix")
CUSTOMER_MIX

### The provider mix - how we pay for COGS

Our customers' terms are irrelevant on this side; what matters is the terms each
provider gives us. This comes in two pieces:

- **`PROVIDERS`** is global - a provider and the term it invoices us on. Defined
  once, because a provider's terms do not change depending on who is buying.
- **`PROVIDER_MIX`** is per country - how COGS spend is split between them, one
  column per country, the same shape as `CUSTOMER_MIX`.

The two are then rolled up into a term-by-country table, so the rest of the
pipeline does not care which stream it is working on.

In [ ]:
require_all_countries(PROVIDER_MIX.columns, "provider mix")

# A provider with no term would be dropped by the rollup below, taking its spend with it.
unknown = sorted(set(PROVIDER_MIX.index) - set(PROVIDERS))
if unknown:
    raise KeyError(f"No payment term defined for provider(s): {unknown}")

# Roll providers up onto their terms, which is what the pipeline works in.
# Two providers sharing a term simply add together.
PROVIDER_TERM_MIX = PROVIDER_MIX.groupby(PROVIDER_MIX.index.map(PROVIDERS)).sum()
PROVIDER_TERM_MIX

## 6. Duty

Duty is charged on the way in and paid on the way out. We pay it on imports, and we
shift it to the customer - but the two figures are not the same, so neither side can
be derived from the other.

That rules out treating duty as a rate. A rate applied to revenue can only ever produce
a duty line that is a fixed fraction of revenue, which is not what the numbers do. So
duty is **budgeted in its own right on each side**, and becomes two more streams:

| | service | duty |
|---|---|---|
| money in | `revenue` | `revenue_duty` |
| money out | `cogs` | `cogs_duty` |

`Revenue Total` is `revenue + revenue_duty` and `COGS Total` is `cogs + cogs_duty`,
which is the way the figures already come.

**What duty is not is a separate clock.** Duty charged to a customer sits on the same
invoice as the service, so it collects on the customer's terms and runs as late as that
customer runs. Duty on the import belongs to the COGS it came with and is paid the same
way. So the two new streams need no new term mix and no new delay table - they point at
the ones already defined, and only the amounts differ.

That means duty costs almost nothing to add here: `STREAMS` gains two entries, which is
the extension section 7 was written for. It also means the old `DUTY_RATE` gross-up
disappears, and with it `cash_share` - every share is now a share of its own stream's
budget, and duty is just two of those streams.

The duty float is still visible, and worth looking for: duty out follows COGS terms and
duty in follows customer terms, so the gap between them is money being financed.

## 7. Four streams

The decomposition is identical for all four - spread the month over work days, split by
payment term, take the due date. Only the mix differs, so the same function runs four
times and each result is tagged with a `stream` column.

The two duty streams reuse their partner's mix outright, because duty rides the invoice
the service is on. That is the whole cost of adding them: two more entries in `STREAMS`,
pointing at tables that already exist.

One code path for all four. If any of them ever needs to diverge - a different invoicing
profile for providers, say - it is `STREAMS` that grows, not `distribute()`.

In [ ]:
def apply_payment_terms(invoiced, mix):
    """Split every invoiced day across its country's terms, and date each split by when it is due."""
    # Normalise each country's column, then reshape the table to one row per country/term.
    weights = ((mix / mix.sum()).stack()
               .rename("weight").rename_axis(["term", "country"]).reset_index())
    weights = weights[weights["weight"] > 0]  # a term nobody uses is not a row worth carrying
    payments = invoiced.merge(weights, on="country")  # each invoiced day x that country's terms
    payments["pct_of_month"] *= payments["weight"]
    payments["due_date"] = [due_date(d, t)
                            for d, t in zip(payments["invoice_date"], payments["term"])]
    return payments.drop(columns="weight")


# Duty shares its partner's mix: it is on the same invoice, so it falls due the same day.
STREAMS = {
    "revenue":      CUSTOMER_MIX,
    "revenue_duty": CUSTOMER_MIX,
    "cogs":         PROVIDER_TERM_MIX,
    "cogs_duty":    PROVIDER_TERM_MIX,
}

payments = pd.concat([apply_payment_terms(invoiced, mix).assign(stream=stream)
                      for stream, mix in STREAMS.items()], ignore_index=True)

payments.head()

In [ ]:
# Each stream redistributes its own monthly budget, so every one of the four still adds
# back up to 100% of it.
(payments.groupby(["stream", "country", "period"])["pct_of_month"].sum()
         .groupby(["stream", "country"]).mean().round(4))

## 8. Payment delays

The due date is when the money is *owed*. What actually happens on the customer
side is that they pay late, and by an amount that varies by country and by term -
a customer on 60-day terms tends to run later than one on 14-day terms, and the
same country can behave differently on each.

So there is one delay table per stream, in the same term-by-country shape as the
mixes: `DELAY_CUSTOMER` for money in, `DELAY_PROVIDER` for money out. The values
are calendar days, like the terms themselves.

`DELAY_PROVIDER` is all zeros for now, so on the COGS side `expected_date` is the
due date and only the work-day roll moves it. The table is there in full shape, so
switching that on later is a change of numbers, not of structure.

This gives three dates per row, and it is worth keeping them apart:

- `due_date` - what the contract says.
- `expected_date` - due date plus the observed delay, still on nominal days.
- `payment_date` - that rolled onto a work day, in the next section.

**One limitation to be aware of.** A single average delay per country and term
shifts a payment without spreading it: a spike that was 20% on one day is still
20% on one day, just later. Real delay statistics have a spread - some pay on
time, a few run very late - and that spread is what smooths a cash peak. If your
statistics carry a distribution rather than just a mean, this is the place to turn
each delay into several weighted buckets, exactly as `TERM_MIX` does for terms.

In [ ]:
def require_full_grid(table, what):
    """A gap in a delay table would leave a payment with no date at all."""
    require_all_countries(table.columns, what)
    missing = [term for term in PAYMENT_TERMS if term not in table.index]
    if missing:
        raise KeyError(f"No {what} defined for term(s): {missing}")


for stream, table in DELAYS.items():
    require_full_grid(table, f"{stream} payment delay")

# Every stream needs a delay table, or apply_delays() would leave its rows undated.
missing = sorted(set(STREAMS) - set(DELAYS))
if missing:
    raise KeyError(f"No payment delay table defined for stream(s): {missing}")

DELAY_CUSTOMER

In [ ]:
def apply_delays(payments, delays=DELAYS):
    """Shift each due date by the delay observed for that stream, country and term."""
    lookup = pd.concat(
        [table.stack().rename("delay_days").rename_axis(["term", "country"])
              .reset_index().assign(stream=stream)
         for stream, table in delays.items()], ignore_index=True)
    out = payments.merge(lookup, on=["stream", "country", "term"], how="left")
    # Calendar days, like the terms themselves - the work-day roll comes after.
    out["expected_date"] = out["due_date"] + pd.to_timedelta(out["delay_days"], unit="D")
    return out.drop(columns="delay_days")


payments = apply_delays(payments)
payments[["stream", "country", "term", "due_date", "expected_date"]].head()

## 9. Back to work days: when the money actually moves

The money only moves on a work day, so an expected date landing on a weekend or a
bank holiday rolls **forward** to the next one.

That closes the loop: invoicing happens on work days, terms and delays are counted
in nominal days, and settlement lands back on work days.

Both streams roll on the country's own calendar. For a provider invoicing from
somewhere else that is an approximation - it is their calendar that would govern -
but it keeps one calendar per country, which is the simpler thing to maintain.

In [ ]:
def to_work_day(country, date):
    """Roll a date forward to the next work day, leaving it alone if it already is one."""
    return CALENDARS[country].rollforward(date)


payments["payment_date"] = [to_work_day(country, date)
                            for country, date in zip(payments["country"], payments["expected_date"])]
payments = payments.sort_values(["stream", "country", "payment_date"]).reset_index(drop=True)
payments.head()

## 10. Factoring

Some countries sell part of their invoicing to a bank, and those invoices settle on a clock
of their own that has nothing to do with the terms in section 5. The week's factored
invoices go out in one run on the Monday after the work. The bank pays out most of the face
value on the Thursday of that run week, whatever term the customer happens to be on - that
is the point of the arrangement, the money arrives on a schedule instead of when somebody
gets round to paying. The rest is held back until the customer has actually paid the bank,
and out of that held-back slice the bank takes its fee.

So one invoice can settle four different ways, and this section gives every row a leg to
say which. The row already carries a share of its stream's month; the leg says which of the
four routes that share travels, and a rate per country and month says how much goes down
each. The direct leg is what the customer still pays us on the old terms, and it is worth
one less the factored share. The advance is the factored share times whatever the bank
advances, and it lands on the Thursday. The retention is what is left of the factored share
once the bank has taken both its advance and its fee, and it keeps the payment date it
always had plus a day for the bank to pass it on. The fee is the factored share times the
bank's rate, on that same day, and it is money going out rather than in.

The four rates add to exactly one, whatever the three parameters are. That is why the fee
is a leg of its own rather than quietly netted off the retention: the shares still add up
to the budget, so the check in section 7 goes on meaning what it meant, and what is
actually received falls out as one less the fee without anybody having to assert it.

Duty is factored with the revenue it rides on. The bank buys the invoice and the duty is a
line on that invoice, so it is sold along with it - the same rule section 6 already applies
to terms and to delays.

The factored slice sits on the same term mix as everybody else. There is no customer
dimension in the model, so a set of factored customers is carried as a share of a country's
revenue rather than as a list of names. If those customers really are on different terms
that is a second mix, not a change to any of this.

The facility is set per country **and per month**. A bank facility is a bargain struck and
struck again rather than settled once, so the share sold, the advance and the fee are read
off the row for the month the invoice belongs to: a country can take a facility on halfway
through the horizon, give it up again, or keep it and sell less through it, and none of
those is a special case anywhere downstream.

What is still decided once for the whole horizon is whether a country carries the four legs
at all, because the rows are written when this notebook runs and only the rates on them
move afterwards. A country that factors in any month of the horizon carries them in all of
them; a month it does not factor in gets nought on three legs and the whole invoice back on
the direct one, which is the same answer as not having the rows. Only a country that never
factors at all can safely be left without them, and in option B not even that.

None of it touches the budget. Factoring is not invoicing - it changes when the money
arrives and how much of it survives the trip, not what was billed - so no P&L line and no
row on the budget sheet moves because of it.

In [ ]:
def month_key(period):
    """How a month is spelled in the factoring tables: the year and the month, nothing else.

    Those tables are typed rather than generated, so their index is a month somebody can
    read and not a timestamp. This is the one place that turns one of the horizon's months
    into the row holding its figures.
    """
    return f"{period:%Y-%m}"


# The horizon, spelled the way the factoring tables are indexed. Read from this rather than
# from a table's own index, so a table left with rows past the end of a horizon that has
# since been shortened contributes nothing rather than quietly extending it again.
FACTORING_MONTHS = [month_key(period) for period in periods]


def require_all_months(table, what):
    """Every month of the horizon needs its own row - a missing one has no figure to read.

    The country check beside it is the same idea one dimension over. Together they are what
    stops a horizon that grew, or a country that was added, from being answered off some
    other month's row instead of raising.
    """
    missing = [month for month in FACTORING_MONTHS if month not in table.index]
    if missing:
        raise KeyError(f"No {what} defined for: {missing}")


for config, what in ((FACTORING, "factoring flag"), (FACTORED_SHARE, "factored share"),
                     (ADVANCE_RATE, "advance rate"), (FACTORING_FEE, "factoring fee")):
    require_all_countries(config.columns, what)
    require_all_months(config, what)




def invoice_run(date):
    """The run that bills a day's work: the first Monday strictly after it.

    Strictly after, because a Monday's run covers the week up to it and not the day it
    goes out on. So a Monday's work waits seven days for the next run and a Friday's
    waits three.
    """
    return date + pd.Timedelta(days=7 - (date.weekday() - INVOICE_RUN_WEEKDAY) % 7)


def advance_date(country, date):
    """When the bank's share of one day's work lands.

    Counted from the nominal Monday rather than a rolled one. A bank holiday on the
    Monday delays the paperwork and not the money, and rolling the run first would push
    a whole week's advance out behind it for no reason. The Thursday itself still rolls,
    because nothing lands on a day the banks are shut.
    """
    return to_work_day(country, invoice_run(date) + pd.Timedelta(days=ADVANCE_OFFSET_DAYS))


def factors_ever(country):
    """Whether the bank buys anything of this country's in any month of the horizon.

    Which legs a country carries is a horizon-wide question even though the rates on them
    are monthly, because the rows are written once, when this notebook runs. A country
    taking a facility on in June needs its advance, retention and fee rows from January
    or June's money would have nowhere to land; the rate is what empties the months before
    it. Only a country answering No in every single month can safely have none at all.
    """
    return bool(FACTORING.loc[FACTORING_MONTHS, country].any())


# leg_rates() is asked the same question once per row of a table of sixty thousand, and
# there are only a few hundred distinct answers to it - four countries by twenty-four
# months by four streams. Each is worked out once and kept rather than read off four
# tables again every time. Keyed on all three, so no answer can outlive a change to any.
LEG_RATE_CACHE = {}


def leg_rates(country, period, stream):
    """What share of an invoice travels each leg. The four of them are the whole invoice.

    It takes the stream and not just the country, because only the streams the bank
    actually buys have four legs. Everything else keeps one, worth the whole invoice,
    whatever the facility says - and so does a stream in a month the country does not
    factor in. Both cases are the model exactly as it was before this section.

    And it takes the month, because a facility is not one bargain struck for the whole
    horizon. The share sold, the advance and the fee are read off the row for the month
    the invoice belongs to, so a country can start factoring, stop, or change what it
    sells partway through without any of it being a special case anywhere downstream.
    """
    cached = LEG_RATE_CACHE.get((country, period, stream))
    if cached is not None:
        return cached

    month = month_key(period)
    if stream not in FACTORED_STREAMS or not FACTORING.at[month, country]:
        rates = {"direct": 1.0, "advance": 0.0, "retention": 0.0, "fee": 0.0}
    else:
        factored = FACTORED_SHARE.at[month, country]
        advance, fee = ADVANCE_RATE.at[month, country], FACTORING_FEE.at[month, country]
        rates = {"direct":    1 - factored,
                 "advance":   factored * advance,
                 "retention": factored * (1 - advance - fee),
                 "fee":       factored * fee}
    LEG_RATE_CACHE[(country, period, stream)] = rates
    return rates


# A fee bigger than what the bank held back would make the retention negative, and a
# leg set that does not add to one invoice would quietly restate the budget. Every month
# as well as every country now: each of the ninety-six rows above can be edited on its
# own, and it is the edited one that will be wrong.
for country in COUNTRIES:
    for period in periods:
        for stream in STREAMS:
            rates = leg_rates(country, period, stream)
            if min(rates.values()) < 0:
                raise ValueError(f"The {stream} legs for {country} in {month_key(period)} "
                                 f"are not all payable: {rates}")
            if abs(sum(rates.values()) - 1) > 1e-12:
                raise ValueError(f"The {stream} legs for {country} in {month_key(period)} "
                                 f"do not add up to one invoice: {rates}")



# The revenue side, which is the only one with anything to show - and only the months in
# which something moved, since a facility left alone all the way through would otherwise
# print the same four numbers twenty-four times over. One row means constant throughout.
legs = pd.DataFrame({(label(country), LEG_LABELS[leg]):
                     [leg_rates(country, period, "revenue")[leg] for period in periods]
                     for country in COUNTRIES for leg in LEG_LABELS},
                    index=FACTORING_MONTHS)
legs[legs.ne(legs.shift()).any(axis=1)]


### Splitting the streams into legs

Every row keeps its whole share of the month and gains a leg; the rate on that leg is
what decides how much of it travels that way. Nothing is subtracted here, which is what
lets the rates live in the workbook rather than in this notebook.

The advance is built from `invoiced` rather than from `payments`, before the split in
section 7 rather than after it. The bank buys the invoice, not the promise behind it, so
what it pays out does not depend on the term the customer is on - and a whole week of
work days ends up on one Thursday.

In [ ]:
# The bank pays on its own clock rather than on the customer's, so these rows have no
# payment term to name. Kept as a value rather than left blank so it reads on the export.
ADVANCE_TERM = "Factored - no term"


def leg_flow(stream, leg):
    """Which side of the cash flow one leg lands on.

    The fee is the odd one out: it hangs off a revenue stream but it is money going out,
    so which side a row falls on cannot be read off its stream alone any more.
    """
    if leg == "fee":
        return "Out"
    return "In" if stream in STREAMS_IN else "Out"


def advance_leg(invoiced, countries):
    """What the bank pays out, taken off the invoiced days rather than the payment terms.

    One row per work day still, not one per Thursday: the two cash sheets group by their
    own date anyway, and keeping the day the work was done means these rows carry an
    invoice date like every other row on the export.
    """
    days = invoiced[invoiced["country"].isin(countries)]
    # Nominal, like every other date in the model before the work-day roll.
    settles = days["invoice_date"].map(invoice_run) + pd.Timedelta(days=ADVANCE_OFFSET_DAYS)
    dated = days.assign(
        term=ADVANCE_TERM, leg="advance",
        # The bank settles on a schedule rather than when it gets round to it, so nothing
        # on this leg is ever owed late: the day it falls due is the day it lands.
        due_date=settles, expected_date=settles,
        payment_date=[advance_date(country, date)
                      for country, date in zip(days["country"], days["invoice_date"])])
    # Duty is on the same invoice, so the bank buys it on the same terms.
    return pd.concat([dated.assign(stream=stream) for stream in FACTORED_STREAMS],
                     ignore_index=True)


def apply_factoring(payments, invoiced):
    """Give every row a leg, and add the three that a factored invoice also settles in."""
    sold = payments["stream"].isin(FACTORED_STREAMS)
    if FACTORING_IS_A_NOTEBOOK_SETTING:
        # Whether the country factors at all, and not whether it factors in this row's
        # month. Dropping the rows for the months it does not would leave a country that
        # switches the facility on partway through with nowhere to put that money; the
        # rate on them is nought in those months, which comes to the same thing and keeps
        # the answer a workbook figure rather than a shape decided here.
        sold &= payments["country"].map(factors_ever)

    # The held-back slice keeps the term and the delay it always had. All that is added is
    # the day the bank takes to pass the money on once the customer has paid it.
    bank = pd.Timedelta(days=BANK_DELAY_DAYS)
    held = payments[sold]
    retention = held.assign(leg="retention", due_date=held["due_date"] + bank,
                            expected_date=held["expected_date"] + bank)
    retention["payment_date"] = [to_work_day(country, date) for country, date
                                 in zip(retention["country"], retention["expected_date"])]

    factoring = [country for country in COUNTRIES
                 if factors_ever(country) or not FACTORING_IS_A_NOTEBOOK_SETTING]
    return pd.concat([payments.assign(leg="direct"),
                      advance_leg(invoiced, factoring),
                      retention,
                      # Taken out of the retention on the day it is released, so the fee
                      # is the same rows over again under a different name.
                      retention.assign(leg="fee")], ignore_index=True)


payments = apply_factoring(payments, invoiced)
payments = payments.sort_values(["stream", "leg", "country", "payment_date"]).reset_index(drop=True)

# The legs of a stream still add up to that stream's month, because the rates for that
# month do. Getting one of them wrong is the way this section would go wrong quietly, so
# it is checked here rather than left to be noticed on a chart - and it is checked row by
# row, which is what catches a rate read off the wrong month as well as a wrong rate.
rated = payments["pct_of_month"] * [
    leg_rates(country, period, stream)[leg] for country, period, stream, leg
    in zip(payments["country"], payments["period"], payments["stream"], payments["leg"])]
totals = rated.groupby([payments["stream"], payments["country"], payments["period"]]).sum()
if not totals.round(9).eq(1).all():
    raise ValueError(f"Legs do not add up to the month:\n{totals[totals.round(9) != 1]}")

payments.groupby(["leg", "stream"]).size().unstack(fill_value=0)


## 11. When the costs move

Revenue and COGS reach cash through a payment term. Everything under them on the P&L does
not. A payroll run goes out on the last working day whatever anybody's terms say, rent on a
date in the lease, an IT bill in dribs across the month. So these lines get a rule instead
of a term.

Five rules cover what is there now. Month end puts all of it on the last working day of the
month. Mid month puts all of it on the 15th, or the next working day if the 15th is not
one. Spread puts it equally across the month's working days. Week puts all of it into one
week of the month, and nothing uses that yet - it is there so a cost that really does land
in the second week has somewhere to go without this section being rewritten for it. And
none means no cash effect at all, which is what bad debt is: written off rather than paid
out.

`COST_TIMING` holds one rule per line per country, in the same line-by-country shape the
mixes in section 5 use, so a country that pays its rent differently is one cell and not a
special case.

These come out at week grain rather than day grain, because that is what the cash flow
sheets read and because it is what makes the rule switchable in the workbook. A month is
cut into the weeks its working days touch, and each week carries five facts: how many of
the month's working days fall in it and how many the month has, whether the last working
day is in it, whether the mid-month day is, and which of the month's weeks it is out of how
many there are. Every one of the five rules is arithmetic over those five facts, so the
rule itself can be a dropdown in Excel with no new rows sitting behind it. The table is
small - four countries by twenty-four months by about five weeks - and it does not depend
on the cost line at all, so one copy of it serves all twelve.

A week that straddles two months appears once under each, holding only that month's days.
The month totals on the cash flow sheets are then real calendar months and tie to the
budget month the money came from, which matters most exactly where a straddling week is
most likely to hurt: the payroll on the last working day.

In [ ]:
require_all_countries(COST_TIMING.columns, "cost timing")

missing = [line for line in SCHEDULED_LINES if line not in COST_TIMING.index]
if missing:
    raise KeyError(f"No cash timing defined for cost line(s): {missing}")
unknown = sorted(set(COST_TIMING.to_numpy().ravel()) - set(TIMING_RULES))
if unknown:
    raise KeyError(f"Not one of the timing rules: {unknown}")




def mid_month_day(country, period):
    """The 15th, or the next work day if the 15th is not one."""
    return to_work_day(country, period + pd.Timedelta(days=MID_MONTH_DAY - 1))


def month_weeks(country, period):
    """One row per week a month's work days touch, and what the rules need to know of it.

    These five facts are between them enough to work out every rule's share, which is what
    lets the rule be a dropdown in the workbook rather than a decision taken here: nothing
    downstream needs a different set of rows when the rule changes, only different weights
    on the rows that are already there.
    """
    days = work_days(country, period)
    mid, last = mid_month_day(country, period), days[-1]
    groups = list(days.isocalendar().groupby(["year", "week"]))
    return [{"country": country, "period": period,
             "week_key": f"{year}-W{week:02d}",
             "week_of_month": index, "weeks_in_month": len(groups),
             "days": len(group), "month_days": len(days),
             "has_last": int(last in group.index),
             "has_mid": int(mid in group.index)}
            for index, ((year, week), group) in enumerate(groups, start=1)]


COST_WEEKS = pd.DataFrame([week for country in COUNTRIES for period in periods
                           for week in month_weeks(country, period)])


def rule_share(rule, chosen, week):
    """What share of a month's cost lands in one of its weeks, under one rule.

    Written out again in Excel against the same five facts, so this and the workbook agree
    by construction rather than by anybody keeping them in step.
    """
    if rule == "none":
        return 0.0
    if rule == "month_end":
        return float(week["has_last"])
    if rule == "mid_month":
        return float(week["has_mid"])
    if rule == "spread":
        return week["days"] / week["month_days"]
    # A month with four weeks and a rule asking for the fifth would lose the cost
    # altogether, so the ask is clamped to the weeks there are.
    return float(week["week_of_month"] == min(chosen, week["weeks_in_month"]))


def cost_cash(weeks=COST_WEEKS, timing=COST_TIMING, chosen=COST_TIMING_WEEK):
    """One row per country, cost line, month and week, and the share landing in it."""
    rows = []
    for week in weeks.to_dict("records"):
        for line in SCHEDULED_LINES:
            rule = timing.at[line, week["country"]]
            rows.append((week["country"], week["period"], line, rule, week["week_key"],
                         week["week_of_month"],
                         rule_share(rule, int(chosen.at[line, week["country"]]), week)))
    return pd.DataFrame(rows, columns=["country", "period", "line", "rule", "week_key",
                                       "week_of_month", "share"])


costs = cost_cash()

# A line pays its month once over, or not at all where the rule says it never reaches cash.
# Anything in between means a rule is dropping money on the floor.
totals = costs.groupby(["country", "period", "line"])["share"].sum().round(9)
paid = costs.groupby(["country", "period", "line"])["rule"].first() != "none"
if not totals.eq(paid.astype(float)).all():
    raise ValueError(f"A month's cost does not add up:\n{totals[totals.ne(paid.astype(float))]}")

costs.groupby(["rule", "line"]).size().unstack(fill_value=0).T


## 12. When the money moves

One invoice month at a time, because shares of different months cannot be added
together without the amounts behind them. Money in is drawn upwards and money out
downwards, and each side is **stacked**: the service first, the duty riding on it in the
lighter shade. The height of a stack is that side's total.

**The four halves are not to scale against each other.** Each is a share of its own
monthly budget, and those are four different numbers. The shapes are comparable; the
heights are not. Netting them into a cash position needs the amounts.

Revenue is spikier than COGS because customers sit across many terms while COGS sits on
just two, so COGS lands late and in two clean blocks. Duty follows its partner exactly -
same days, same spikes - so the pairs move together and only the size of the lighter
band changes.

In [ ]:
month = payments["period"].min()  # change this to look at another invoice month
subset = payments[payments["period"] == month].copy()

# What a row is actually worth once the rate its leg carries in this month is applied.
# Adding the raw shares across legs would count a factored invoice four times over. The
# fee goes in negative rather than becoming a stream of its own, so a revenue bar here is
# net of the bank's cut. The month is `month`: every row in the subset is one of its own.
subset["cash"] = [share * leg_rates(country, month, stream)[leg] * (-1 if leg == "fee" else 1)
                  for country, stream, leg, share
                  in zip(subset["country"], subset["stream"], subset["leg"],
                         subset["pct_of_month"])]

SIGN = {**{stream: 1 for stream in STREAMS_IN},      # cash in up, cash out down
        **{stream: -1 for stream in STREAMS_OUT}}
# Colour means side; the lighter half of each pair is the duty riding on it.
STREAM_COLOUR = {"revenue": PALETTE[0], "revenue_duty": tint(PALETTE[0]),
                 "cogs": PALETTE[1], "cogs_duty": tint(PALETTE[1])}
ORDER = list(STREAMS_IN) + list(STREAMS_OUT)

by_country = list(subset.groupby("country"))
fig, axes = plt.subplots(len(by_country), 1, figsize=(12, 2.3 * len(by_country)),
                         sharex=True, sharey=True, constrained_layout=True)

for ax, (country, rows) in zip(axes, by_country):
    # A common date index per country, so the duty bar can be stacked on the service one.
    landing = (rows.pivot_table(index="payment_date", columns="stream",
                                values="cash", aggfunc="sum")
                   .reindex(columns=ORDER).fillna(0) * 100)
    for side in (STREAMS_IN, STREAMS_OUT):
        bottom = 0
        for stream in side:
            values = landing[stream] * SIGN[stream]
            ax.bar(landing.index, values, bottom=bottom,
                   color=STREAM_COLOUR[stream], width=1.0)
            bottom = bottom + values
    ax.axhline(0, color=RULE, linewidth=1)
    style(ax, label(country), INK)  # colour is carrying the stream, so the title stays plain ink

handles = [plt.Rectangle((0, 0), 1, 1, color=STREAM_COLOUR[stream]) for stream in ORDER]
fig.legend(handles, ["revenue in", "duty in", "COGS out", "duty out"],
           loc="outside upper right", frameon=False, ncols=4)
fig.suptitle(f"When {month:%B %Y} turns into cash", fontsize=13, x=0.01, ha="left")
fig.supylabel("% of that stream's monthly budget", fontsize=10, color=MUTED)
plt.show()

## 13. Export

`with_labels()` swaps the full country name for its short code on the way out, so
the data keeps the spelled-out name and only what leaves the notebook is
abbreviated.

In [ ]:
export = with_labels(payments)[
    ["stream", "leg", "country", "period", "invoice_date", "term",
     "due_date", "expected_date", "payment_date", "pct_of_month"]
]
export.head()

## 14. Excel export

One workbook, holding the budget, the metrics behind the forecast, and the three results
worth looking at:

1. **Share of the month per work day** - the invoicing profile from section 4.
2. **Due date cash** - what is owed when, for both streams, before any delay.
3. **Payment date cash** - the same after the delays and the work-day roll.

Only the third becomes the cash flow model. The other two are there to show how it
was arrived at, which is most of what makes a forecast arguable rather than magic.

Built one sheet at a time, starting with the sheet the budget is typed on - because
everything else in the workbook now reads it.

In [ ]:
from openpyxl import Workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.chart.descriptors import NumberFormatDescriptor
from openpyxl.chart.label import DataLabel, DataLabelList
from openpyxl.chart.shapes import GraphicalProperties
from openpyxl.descriptors.nested import NestedBool, NestedInteger
from openpyxl.chart.text import RichText
from openpyxl.drawing.text import (CharacterProperties, Paragraph, ParagraphProperties,
                                   RichTextProperties)
from openpyxl.styles import Alignment, Border, Font, Protection, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.worksheet.table import Table, TableStyleInfo



# Same hue as the service it rides on, lighter, so a pair reads as one side of the flow.
STREAM_COLOURS = {"revenue": PALETTE[0], "revenue_duty": tint(PALETTE[0]),
                  "cogs": PALETTE[1], "cogs_duty": tint(PALETTE[1])}

# Everything the budget sheet carries a row for: the four traded streams, which reach cash
# through a payment term, and the twelve scheduled lines, which reach it on a rule. Past
# the budget they are the same thing - a name, a month and an amount - so anything looking
# an amount up reads this rather than one of the two halves.
CATEGORY_LABELS = {**STREAM_LABELS, **SCHEDULED_LINES}


def rgb(colour):
    """openpyxl wants bare hex; the palette is written the way matplotlib wants it."""
    return colour.lstrip("#").upper()


# The workbook borrows the palette the charts already use, so the two read as one document.
FACE = "Arial"
TITLE = Font(name=FACE, size=12, bold=True, color=rgb(INK))
HEADER = Font(name=FACE, size=10, bold=True, color=rgb(INK))
BODY = Font(name=FACE, size=10, color=rgb(INK))
ENTRY = Font(name=FACE, size=10, color=rgb(PALETTE[0]))  # blue is the convention for "type here"
NOTE = Font(name=FACE, size=9, italic=True, color=rgb(MUTED))

UNDERLINE = Border(bottom=Side(style="thin", color=rgb(RULE)))

# Row 1 and column A are margin: thin, and never written to. The title goes in row 2,
# row 3 takes a note where a sheet needs one, and the table starts its headers in row 4.
MARGIN_HEIGHT, MARGIN_WIDTH = 6, 2.5
TITLE_ROW, NOTE_ROW, HEADER_ROW = 2, 3, 4
FIRST_DATA_ROW = HEADER_ROW + 1


def new_sheet(book, name, widths):
    """A blank sheet in the house style. `widths` sizes the content columns, left to right from B.

    A sheet of the same name is replaced, so re-running a cell rewrites its sheet
    rather than leaving a second copy behind it.
    """
    if name in book.sheetnames:
        del book[name]
    sheet = book.create_sheet(name)
    sheet.sheet_view.showGridLines = False
    sheet.row_dimensions[1].height = MARGIN_HEIGHT
    sheet.column_dimensions["A"].width = MARGIN_WIDTH
    for offset, width in enumerate(widths):
        sheet.column_dimensions[get_column_letter(2 + offset)].width = width
    return sheet


def write_title(sheet, text):
    """The sheet's name, in its top-left corner."""
    sheet.cell(row=TITLE_ROW, column=2, value=text).font = TITLE


def write_note(sheet, text):
    """A line under the title, for something a sheet needs said before it is read."""
    sheet.cell(row=NOTE_ROW, column=2, value=text).font = NOTE


def cell_value(value):
    """Python natives only - openpyxl has nothing to do with a numpy scalar."""
    if isinstance(value, pd.Timestamp):
        return value.to_pydatetime()
    return value.item() if hasattr(value, "item") else value


def column_of(columns, header, first_column=2):
    """The sheet column letter that one of a table's columns ends up in."""
    return get_column_letter(first_column
                             + [name for name, _, _, _ in columns].index(header))


def write_table(sheet, table_name, columns, rows, font=BODY, first_column=2):
    """Headers, the rows under them, and a real Excel table over the two.

    `columns` is (header, width, number format, alignment) in order, and `rows` is an
    iterable of tuples in that same order. Being a named table rather than a loose
    block is what makes a sheet worth importing: a filter, a pivot or a query picks
    the range up whole, and keeps working when the number of rows changes.

    `first_column` puts a second table beside the first on the same sheet, sharing its
    header row. Only the one in the usual place freezes the panes - two of them would
    fight over where the split goes.
    """
    for offset, (header, _, _, align) in enumerate(columns):
        cell = sheet.cell(row=HEADER_ROW, column=first_column + offset, value=header)
        cell.font = HEADER
        cell.border = UNDERLINE
        cell.alignment = Alignment(horizontal=align)

    written = 0
    for written, values in enumerate(rows, start=1):
        for offset, (_, _, number_format, align) in enumerate(columns):
            cell = sheet.cell(row=HEADER_ROW + written, column=first_column + offset,
                              value=cell_value(values[offset]))
            cell.font = font
            cell.number_format = number_format
            cell.alignment = Alignment(horizontal=align)

    first_letter = get_column_letter(first_column)
    last_column = get_column_letter(first_column - 1 + len(columns))
    table = Table(displayName=table_name,
                  ref=f"{first_letter}{HEADER_ROW}:{last_column}{HEADER_ROW + written}")
    # No banding: the point of the table here is the named range, not the decoration.
    table.tableStyleInfo = TableStyleInfo(name="TableStyleLight1", showRowStripes=False)
    sheet.add_table(table)
    if first_column == 2:
        sheet.freeze_panes = f"B{FIRST_DATA_ROW}"
    return written


def add_dropdown(sheet, column, choices, first_row, last_row):
    """A list dropdown down one column, so a value gets picked instead of typed and mistyped."""
    allowed = [str(choice) for choice in choices]
    rule = DataValidation(type="list", formula1='"' + ",".join(allowed) + '"',
                          allow_blank=True, showErrorMessage=True)
    rule.errorTitle, rule.error = "Not one of the options", "Pick one of: " + ", ".join(allowed)
    sheet.add_data_validation(rule)
    rule.add(f"{column}{first_row}:{column}{last_row}")

In [ ]:
# One panel per country, stacked down the right of the table - the same decision the
# charts in sections 4 and 10 make, and for the same reason. A country is the level at
# which a percentage means something: every figure here is a share of one country's
# budget, so four of them on one pair of axes would either be four series fighting over
# the same space or, worse, one series adding four things that cannot be added. That
# holds whatever currency they are in; every budget here is in DKK, and a share of one
# country's budget still means nothing beside a share of another's.
#
# Each panel carries the small block of numbers it reads, summed out of the table with
# SUMIFS. A chart needs a compact rectangle and these tables run to thousands of rows.



# Panels start below the frozen rows. A chart anchored inside them is pinned to the
# pane and sits over the table for the whole of the scroll, rather than moving with
# the numbers it belongs to.
PANEL_FIRST_ROW = FIRST_DATA_ROW + 1


class HiddenLabel(DataLabel):
    """One data label switched off.

    openpyxl models `delete` on the label *list*, where it hides every label at once,
    and not on the individual label - so the one element that suppresses a single
    point is added back here. The alternative, a label with all its show flags off,
    leaves an empty text box behind rather than nothing.
    """

    # Both descriptors are redeclared, not just the new one: openpyxl works out which
    # fields are nested elements from the class's own namespace, so an inherited `idx`
    # would not be recognised as one and would fail to serialise.
    idx = NestedInteger()
    delete = NestedBool(allow_none=True)
    __elements__ = ("idx", "delete")

    def __init__(self, idx=0):
        self.delete = True
        super().__init__(idx=idx)


class LabelList(DataLabelList):
    """A label list whose number format is actually honoured.

    openpyxl writes `<numFmt formatCode="..."/>` and no `sourceLinked`, which Excel
    reads as "use the cell's own format" - so the format asked for here is ignored and
    the label comes out formatted like the cell behind it. Swapping in the descriptor
    the axes already use writes the attribute, and the format takes effect.
    """

    numFmt = NumberFormatDescriptor()
    __elements__ = DataLabelList.__elements__
    __nested__ = tuple(name for name in DataLabelList.__nested__ if name != "numFmt")


def label_every(weeks):
    """How many bars to step between labels, so a chart carries about TARGET_LABELS."""
    return max(1, round(len(weeks) / TARGET_LABELS))


def value_labels(values, number_format, step, scale=None, size=650):
    """Value labels turned on their side, thinned twice over.

    A hundred weeks across, a label on every bar is a wall of text. Turning them
    upright costs a tenth of the width; printing only every `step`-th bar - the same
    ones the axis names - thins it again; and any bar shorter than LABEL_FLOOR of the
    tallest is left bare, because a number floating over a sliver of a bar is the
    untidiest thing on a chart. `scale` sets what it is measured against, so every
    series in a panel is judged by the same yardstick.
    """
    labels = LabelList()
    labels.showVal = True
    for hide in ("showSerName", "showCatName", "showLegendKey", "showPercent", "showBubbleSize"):
        setattr(labels, hide, False)
    labels.numFmt = number_format
    shape = GraphicalProperties(noFill=True)
    shape.line.noFill = True          # nothing drawn round the number
    labels.spPr = shape
    small = CharacterProperties(sz=size)
    labels.txPr = RichText(bodyPr=RichTextProperties(rot=LABEL_ROTATION, vertOverflow="overflow"),
                           p=[Paragraph(pPr=ParagraphProperties(defRPr=small), endParaRPr=small)])

    reach = scale if scale is not None else max((abs(value) for value in values), default=0)
    floor = reach * LABEL_FLOOR
    labels.dLbl = [HiddenLabel(idx=index) for index, value in enumerate(values)
                   if index % step or abs(value) <= floor]
    return labels


def week_chart(title, y_title, number_format, step):
    """A long, low column chart of something by ISO week, in the house style."""
    chart = BarChart()
    chart.type, chart.grouping, chart.overlap = "col", "stacked", 100
    chart.title = title
    chart.width, chart.height = CHART_WIDTH, CHART_HEIGHT
    chart.x_axis.title, chart.y_axis.title = "ISO week", y_title
    chart.y_axis.numFmt = number_format
    # No gridlines. There is no visible scale to read them against, so they were ruling
    # off empty space; the value labels on the bars carry the numbers instead.
    chart.y_axis.majorGridlines = None
    chart.x_axis.tickLblPos = "low"    # labels under the plot, clear of the bars below zero
    chart.x_axis.tickLblSkip = chart.x_axis.tickMarkSkip = step
    return chart


def add_week_panels(sheet, columns, row_count, weeks, panels, measure, chart_title,
                    y_title, net=None):
    """A stack of small charts, one per panel, each beside the numbers it is drawn from.

    `panels` is (panel name, series, net values), and each series is
    (label, {table column: value to match}, colour, sign, values). A sign of -1 draws
    that series below zero, which is how money out is told from money in. Series are
    stacked, so a service and the duty riding on it make one bar whose height is the
    total for that side.

    The values ride along with the criteria only so the labels can be thinned. The
    cells themselves are always SUMIFS, so the charts stay live.

    `net` is (measure column, chart title, axis title) and adds a second chart under
    each panel's first, collapsing the series to one signed line. It takes a measure of
    its own on purpose: netting money in against money out is only meaningful in money.
    The shares are shares of four different budgets, so their signed sum means nothing.
    """
    def table_range(header):
        letter = column_of(columns, header)
        return f"${letter}${FIRST_DATA_ROW}:${letter}${FIRST_DATA_ROW + row_count - 1}"

    def criteria_of(criteria):
        return "".join(f',{table_range(header)},"{value}"' for header, value in criteria.items())

    measure_range, week_range = table_range(measure), table_range("Week key")
    start = 2 + len(columns) + CHART_GAP
    week_letter = get_column_letter(start)
    widest = max(len(series) for _, series, _ in panels)
    net_column = start + 1 + widest if net else None
    last_column = net_column or start + widest
    step = label_every(weeks)

    # Tall enough for the numbers or for the charts beside them, whichever needs more room.
    block_height = max(len(weeks) + 1, CHART_ROWS * (2 if net else 1)) + PANEL_GAP

    for index, (name, series, net_values) in enumerate(panels):
        top = index * block_height + PANEL_FIRST_ROW
        headers = ["Week", *(label for label, *_ in series)] + (["Net"] if net else [])
        for offset, header in enumerate(headers):
            cell = sheet.cell(row=top, column=start + offset, value=header)
            cell.font, cell.border = HEADER, UNDERLINE
            cell.alignment = Alignment(horizontal="center" if offset == 0 else "right")

        for position, week in enumerate(weeks):
            row = top + 1 + position
            cell = sheet.cell(row=row, column=start, value=week)
            cell.font, cell.number_format = BODY, "@"
            cell.alignment = Alignment(horizontal="center")
            for offset, (_, criteria, _, sign, _) in enumerate(series):
                cell = sheet.cell(row=row, column=start + 1 + offset)
                cell.value = (f"={'-' if sign < 0 else ''}SUMIFS({measure_range},"
                              f"{week_range},${week_letter}{row}{criteria_of(criteria)})")
                cell.font, cell.number_format = BODY, "0.0%"
            if net:
                # Its own SUMIFS over the money column rather than a sum of the cells
                # above, which are shares and must not be added across streams.
                net_range = table_range(net[0])
                terms = "".join(
                    f"{'-' if sign < 0 else '+'}SUMIFS({net_range},{week_range},"
                    f"${week_letter}{row}{criteria_of(criteria)})"
                    for _, criteria, _, sign, _ in series)
                cell = sheet.cell(row=row, column=net_column)
                cell.value = "=" + terms.lstrip("+")
                cell.font, cell.number_format = BODY, "#,##0"

        last = top + len(weeks)
        # One yardstick for the whole panel, so a short bar stays bare even where its
        # own series happens to be short everywhere.
        scale = max((abs(value) for _, _, _, _, values in series for value in values),
                    default=0)
        chart = week_chart(f"{name} - {chart_title}", y_title, "0%;0%", step)
        chart.add_data(Reference(sheet, min_col=start + 1, max_col=start + len(series),
                                 min_row=top, max_row=last), titles_from_data=True)
        chart.set_categories(Reference(sheet, min_col=start, min_row=top + 1, max_row=last))
        for plotted, (_, _, colour, _, values) in zip(chart.series, series):
            plotted.graphicalProperties.solidFill = rgb(colour)
            plotted.graphicalProperties.line.noFill = True   # no outline; the fill carries it
            # Whole percents: a decimal place doubles the width of every label to say
            # something the eye cannot pick out of a bar this size anyway.
            plotted.dLbls = value_labels(values, "0%", step, scale=scale)
        if len(series) == 1:
            chart.legend = None              # a legend naming one thing says nothing
        sheet.add_chart(chart, f"{get_column_letter(last_column + 2)}{top}")

        if net:
            _, net_title, net_y_title = net
            below = week_chart(f"{name} - {net_title}", net_y_title, "#,##0;(#,##0)", step)
            below.legend = None
            below.add_data(Reference(sheet, min_col=net_column, min_row=top, max_row=last),
                           titles_from_data=True)
            below.set_categories(Reference(sheet, min_col=start, min_row=top + 1, max_row=last))
            plotted = below.series[0]
            plotted.graphicalProperties.solidFill = rgb(PALETTE[2])
            plotted.graphicalProperties.line.noFill = True
            # Thousands: the trailing comma in the format divides by 1000, so a net of
            # 917,717 prints as 918k and fits between two bars.
            plotted.dLbls = value_labels(net_values, '#,##0,"k"', step)
            sheet.add_chart(below, f"{get_column_letter(last_column + 2)}{top + CHART_ROWS}")

    for column in range(start, last_column + 1):
        sheet.column_dimensions[get_column_letter(column)].width = 13


### The budget sheets: profit and loss, month by month

The sheet the budget is actually kept on, laid out the way it is kept: twelve months
across, the profit and loss down the side, and the year in the last column. **One per
country per year** - four countries and two years, so eight of them, `P&L DNK 2026`
through `P&L USA 2027`.

This is the input the rest of the workbook now hangs off. Type a figure here and the
budget sheet below picks it up, and every cash row that reads the budget moves with it.

**Costs go in negative.** COGS, manpower and SG&A alike. That is why every subtotal here
is an addition and never a subtraction: gross profit is revenue *plus* a negative cost,
and EBITDA is gross profit 2 *plus* a negative SG&A. Type a cost in as a positive and the
sheet reports it as income without complaining.

**The header is three rows.** Row 4 is the year, merged across the twelve months. Row 5
says whether a column is an actual or a budget - only the first year has anything closed
on it, and the same rule sets the `Actuals/Budget` column on the budget sheet. Row 6
carries the month names, with the unit sitting in the corner where a heading for the
label column would go. Every sheet is kept in **DKK thousands**, whichever country it is
for, so the workbook has one reporting currency and the eight of them add up. What it does
not hold is the rate a country's budget was translated at - that happens before a figure
is typed here.

**Column O is a hidden spacer.** It is what keeps `Total` in column P out of the month
block: a `SUM` over January to December stops at N and cannot reach the total, so the
total column can never end up inside its own sum.

**Six rows carry nothing and are hidden** - Depreciation, Amortization, EBIT, Financial
Income, Financial Expenses and Net Profit before tax. They are written out all the same,
blanks and all, because taking them out would move every row under them somewhere else -
and a budget pasted in from outside has to land line for line. `PL_SHOW_UNUSED` unhides
them.

The blank rows between blocks are part of the layout rather than padding left over from
building it, so they are written out as blanks and everything below them is numbered
around them.

**Every row is declared once**, in `PL_ROWS`: an id, the label that shows in column B,
what kind of row it is, and - for the derived ones - which rows it reads. The formulas are
built from those ids, so `Gross Profit 1 - Courier` is written as "revenue courier plus
COGS courier" and the row numbers are worked out from the layout afterwards. Move a block
or insert a line and the formulas follow it; not one cell address is typed in here.

A contiguous run of rows comes out as `SUM(top:bottom)` - the shape expected under a block
of costs - and anything else as its terms added one by one, so `Gross Profit 1 - Courier`
reads `=C9+C16` and `Revenue total` reads `=SUM(C9:C11)`.

### The figures on them

All eight arrive pre-filled. Every typed line is a fraction of that month's courier
revenue, seasonally shaped and jittered, which lands the sheets at roughly a 26% gross
margin and an 8% EBITDA margin - close enough to a P&L to read like one, and not close
enough to anything to be believed. Replace the numbers.

The seed is the sheet itself, `country-year`, rather than one generator running through
all eight in order. So a figure depends only on which sheet it is on: add a country, or
build the years the other way round, and nothing already in the workbook moves.

One thing worth flagging: the block is headed *Ratios (excl. Duty)* and the two gross
margins are - but both EBITDA margins divide by `Revenue total`, which includes duty. That
is how it was specified, so that is how it is built.

In [ ]:
import random

PL_SHEET_PREFIX = "P&L"            # a tab is "P&L DNK 2026"; the full title goes in B2
PL_TITLE = "Profit and Loss, Month by month"
PL_MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
             "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# The horizon, declared here because this is where the budget is typed. The budget input
# sheet below reads both of these rather than keeping a second copy of them. The years are
# taken from the horizon rather than listed again - one P&L per country per year, and no
# second list that can fall behind START and HORIZON_MONTHS.
BUDGET_YEARS = tuple(range(periods.min().year, periods.max().year + 1))


# B holds the labels, C to N the twelve months, O is a spacer that stays hidden, and P is
# the year. The spacer is what keeps the total out of the month block: a SUM across the
# months stops at N, so the total column can never be caught inside its own sum.
PL_LABEL_COLUMN = 2
PL_FIRST_MONTH_COLUMN = 3
PL_SPACER_COLUMN = PL_FIRST_MONTH_COLUMN + len(PL_MONTHS)
PL_TOTAL_COLUMN = PL_SPACER_COLUMN + 1

PL_YEAR_ROW, PL_BASIS_ROW, PL_HEADER_ROW = 4, 5, 6
PL_FIRST_ROW = PL_HEADER_ROW + 2   # row 7 is left clear, so the first line lands on row 8
PL_WIDTHS = [34] + [11] * len(PL_MONTHS) + [2, 12]

# The rows that carry no figures are hidden, the way they are on the sheet this copies.
# They are written out all the same, because taking them out would move everything under
# them onto a different row. Set this True to see them.
PL_SHOW_UNUSED = False

# A dash for zero, so an empty line reads as empty rather than as a nought, and negatives
# in brackets, which is how a cost is read on a P&L.
PL_MONEY = '#,##0;(#,##0);"-"'
PL_PERCENT = '0.0%;(0.0%);"-"'

YEAR_FONT = Font(name=FACE, size=11, bold=True, color=rgb(INK))
TOP_RULE = Border(top=Side(style="thin", color=rgb(RULE)))

# id, label, kind, what it reads. The ids are what the formulas are written in terms of,
# so nothing below has to know a row number - the layout works them out.
#
#   title  - a heading, nothing beside it     input  - typed in, in blue
#   calc   - a formula, plain                 total  - a formula, ruled and bold
#   ratio  - (numerator, denominator)         blank  - a spacer row, deliberately empty
#   unused - carries no figures, and is hidden; the row below it is still where it was
#   gap    - a spacer inside a hidden block, hidden with it
#
# Costs are entered negative, which is why every subtotal here is an addition.
PL_ROWS = [
    ("revenue",       "Revenue",                      "title", None),
    ("rev_courier",   "Courier",                      "input", None),
    ("rev_ff",        "Freight Forward segment",      "input", None),
    ("rev_duty",      "Duty",                         "input", None),
    ("rev_total",     "Revenue total",                "total", ("rev_courier", "rev_ff",
                                                                "rev_duty")),
    ("rev_ex_duty",   "Revenue total ex. duty",       "total", ("rev_courier", "rev_ff")),
    (None,            "",                             "blank", None),

    ("cogs",          "Cost of Goods sold",           "title", None),
    ("cogs_courier",  "Courier",                      "input", None),
    ("cogs_ff",       "Freight forward segment",      "input", None),
    ("cogs_duty",     "Duty",                         "input", None),
    ("cogs_total",    "Cost of Goods sold",           "total", ("cogs_courier", "cogs_ff",
                                                                "cogs_duty")),
    (None,            "",                             "blank", None),

    ("gp1",           "Gross Profit 1",               "title", None),
    ("gp1_courier",   "Courier",                      "calc",  ("rev_courier", "cogs_courier")),
    ("gp1_ff",        "Freight Forward segment",      "calc",  ("rev_ff", "cogs_ff")),
    ("gp1_duty",      "Duty",                         "calc",  ("rev_duty", "cogs_duty")),
    ("gp1_total",     "Gross Profit 1 total",         "total", ("gp1_courier", "gp1_ff",
                                                                "gp1_duty")),
    (None,            "",                             "blank", None),

    ("manpower",      "Manpower",                     "title", None),
    ("mp_courier",    "Courier segment",              "input", None),
    ("mp_ff",         "Freight Forward segment",      "input", None),
    ("mp_total",      "Manpower total",               "total", ("mp_courier", "mp_ff")),
    (None,            "",                             "blank", None),

    ("gp2",           "Gross Profit 2",               "title", None),
    ("gp2_courier",   "Courier segment",              "calc",  ("gp1_courier", "mp_courier")),
    ("gp2_ff",        "Freight Forward segment",      "calc",  ("gp1_ff", "mp_ff")),
    # Nothing is charged to duty between the two, so this is GP1's duty carried down.
    ("gp2_duty",      "Duty",                         "calc",  ("gp1_duty",)),
    ("gp2_total",     "Gross Profit 2 total",         "total", ("gp2_courier", "gp2_ff",
                                                                "gp2_duty")),
    (None,            "",                             "blank", None),

    ("sga",           "SG&A",                         "title", None),
    ("sga_staff",     "Staff cost",                   "input", None),
    ("sga_vehicle",   "Vehicle",                      "input", None),
    ("sga_rent",      "Rent",                         "input", None),
    ("sga_it",        "IT",                           "input", None),
    ("sga_sales",     "Sales & Marketing",            "input", None),
    ("sga_travel",    "Travel & Rep",                 "input", None),
    ("sga_admin",     "Admin & Misc",                 "input", None),
    ("sga_bad_debt",  "Bad debt",                     "input", None),
    ("sga_sla",       "SLA",                          "input", None),
    ("sga_external",  "External Assistance",          "input", None),
    ("sga_total",     "Total SG&A",                   "total", ("sga_staff", "sga_vehicle",
                                                                "sga_rent", "sga_it",
                                                                "sga_sales", "sga_travel",
                                                                "sga_admin", "sga_bad_debt",
                                                                "sga_sla", "sga_external")),
    (None,            "",                             "blank", None),

    ("ebitda",        "EBITDA",                       "total", ("gp2_total", "sga_total")),
    (None,            "",                             "blank", None),

    ("oneoff",        "Normalized one-offs",            "input", None),
    ("oneoff_hard",   "Normalized one-offs Hardclose",  "input", None),
    ("oneoff_stream", "Normalized one-offs Streamline", "input", None),
    (None,            "",                               "blank", None),

    ("norm_ebitda",   "Normalized EBITDA",            "total", ("ebitda", "oneoff",
                                                                "oneoff_hard",
                                                                "oneoff_stream")),
    (None,            "",                             "blank", None),

    # Nothing is booked on these six, and they are hidden. They stay in the layout so
    # everything below them keeps the row it has always had, and so a budget pasted in
    # from the template outside lands line for line.
    ("depreciation",  "Depreciation",                 "unused", None),
    ("amortization",  "Amortization",                 "unused", None),
    (None,            "",                             "gap",    None),
    ("ebit",          "EBIT",                         "unused", None),
    (None,            "",                             "gap",    None),
    ("fin_income",    "Financial Income",             "unused", None),
    ("fin_expenses",  "Financial Expenses",           "unused", None),
    (None,            "",                             "gap",    None),
    ("npbt",          "Net Profit before tax",        "unused", None),
    (None,            "",                             "gap",    None),

    # The plain EBITDA, not the normalized one: the one-offs are taken out to judge the
    # business by, and they were still cash.
    ("cf_operating",  "Cash flow from oper. activ.",  "calc",  ("ebitda",)),
    ("cf_investment", "Cash flow investment",         "input", None),
    ("cf_financing",  "Cash flow Financing",          "input", None),
    ("cf_net",        "Net Cash Flow",                "total", ("cf_operating", "cf_investment",
                                                                "cf_financing")),
    (None,            "",                             "blank", None),

    ("ratios",        "Ratios (excl. Duty)",          "title", None),
    ("gm_courier",    "Gross Margin Courier",         "ratio", ("gp1_courier", "rev_courier")),
    ("gm_ff",         "Gross Margin Freight Forward", "ratio", ("gp1_ff", "rev_ff")),
    # Both EBITDA margins are measured against revenue *including* duty, unlike the two
    # gross margins above them and unlike what the heading on the block says.
    ("ebitda_margin", "EBITDA margin",                "ratio", ("ebitda", "rev_total")),
    ("norm_margin",   "Normalized EBITDA margin",     "ratio", ("norm_ebitda", "rev_total")),
]


def pl_row_numbers(rows, first_row):
    """Where each id ends up, once the blanks in the layout are counted in."""
    return {key: first_row + offset for offset, (key, *_) in enumerate(rows) if key}


PL_ROW_AT = pl_row_numbers(PL_ROWS, PL_FIRST_ROW)


def pl_sheet_name(country, year):
    """`P&L DNK 2026`.

    The budget sheet rebuilds this name in Excel out of a row's own country and year, so
    the three pieces and the spaces between them are load-bearing.
    """
    return f"{PL_SHEET_PREFIX} {label(country)} {year}"


def pl_sum(letter, keys):
    """Adding a set of rows up, in one column.

    A contiguous block goes out as SUM(top:bottom) - the shape expected under a block of
    costs - and anything else as its terms added one by one, because a SUM over a range
    that skips rows would be a lie about what is in it.
    """
    numbers = sorted(PL_ROW_AT[key] for key in keys)
    if len(numbers) > 1 and numbers == list(range(numbers[0], numbers[-1] + 1)):
        return f"=SUM({letter}{numbers[0]}:{letter}{numbers[-1]})"
    return "=" + "+".join(f"{letter}{number}" for number in numbers)


def pl_ratio(letter, rule):
    """One row over another, blank rather than #DIV/0! where the line below is empty."""
    numerator, denominator = rule
    return f'=IFERROR({letter}{PL_ROW_AT[numerator]}/{letter}{PL_ROW_AT[denominator]},"")'


# --- placeholder figures -------------------------------------------------------------
# Stand-ins, so eight sheets arrive looking like eight P&Ls rather than eight empty
# grids. Every typed line is a fraction of that month's courier revenue - the one figure
# the shape is hung off - and the four calculated blocks come out at roughly a 26% gross
# margin and an 8% EBITDA margin. Nobody should believe any of it. Replace the numbers.
PL_SEED = 20260101

# Years the P&L sheets are left empty. The layout, the formulas and every sheet that
# reads them are all still there - only the typed figures are nought, so a year can be
# planned later without anything downstream having to be rebuilt for it.
BUDGET_ZERO_YEARS = (2027,)

PL_BASE = {              # a plausible month of courier revenue, in DKK thousands
    "Denmark": 4_000,
    "Norway": 3_200,
    "Sweden": 5_100,
    "United States": 8_600,
}
require_all_countries(PL_BASE, "P&L base revenue")

# January to December, so a row is not a wall of one number.
PL_SEASONALITY = [0.95, 0.92, 1.05, 1.00, 1.03, 1.08, 0.80, 0.78, 1.10, 1.12, 1.06, 0.90]
PL_JITTER = 0.08         # how far a month wanders off its seasonal shape

# Every typed row, as a fraction of that month's courier revenue and the sign it carries.
# The two duty lines are deliberately not equal, because a duty recharged and a duty paid
# that always matched would not need two budget lines in the first place.
PL_SHAPE = {
    "rev_courier":   (1.000, +1),
    "rev_ff":        (0.550, +1),
    "rev_duty":      (0.350, +1),
    "cogs_courier":  (0.700, -1),
    "cogs_ff":       (0.440, -1),
    "cogs_duty":     (0.330, -1),
    "mp_courier":    (0.090, -1),
    "mp_ff":         (0.050, -1),
    "sga_staff":     (0.060, -1),
    "sga_vehicle":   (0.012, -1),
    "sga_rent":      (0.018, -1),
    "sga_it":        (0.010, -1),
    "sga_sales":     (0.008, -1),
    "sga_travel":    (0.005, -1),
    "sga_admin":     (0.007, -1),
    "sga_bad_debt":  (0.003, -1),
    "sga_sla":       (0.004, -1),
    "sga_external":  (0.006, -1),
    "oneoff":        (0.004, +1),
    "oneoff_hard":   (0.003, +1),
    "oneoff_stream": (0.002, +1),
    "cf_investment": (0.020, -1),
    "cf_financing":  (0.010, +1),
}


def pl_amounts(country, year):
    """The typed figures for one sheet: a number per typed row and month.

    Seeded off the sheet itself rather than off one generator running through all eight,
    so a figure depends only on which sheet it is on. Add a country, or build the years in
    another order, and nothing already in the workbook moves.

    A year in BUDGET_ZERO_YEARS comes back empty instead. The sheet is still written and
    every formula on it still works - there is simply nothing typed on it yet.
    """
    if year in BUDGET_ZERO_YEARS:
        return {key: [0] * len(PL_MONTHS) for key in PL_SHAPE}
    rng = random.Random(f"{PL_SEED}-{country}-{year}")
    base = PL_BASE[country]
    return {key: [sign * round(base * PL_SEASONALITY[month] * fraction
                               * rng.uniform(1 - PL_JITTER, 1 + PL_JITTER))
                  for month in range(len(PL_MONTHS))]
            for key, (fraction, sign) in PL_SHAPE.items()}


# --- what the budget sheet reads off a P&L -------------------------------------------
# Which lines each budget stream is, and the signs that turn them into positive amounts:
# money out is signed downstream, by the charts and the net column, not here. COGS is the
# cost line with the duty taken back out of it, because duty is budgeted as a stream of
# its own and would otherwise be counted twice.
PL_TO_BUDGET = {
    "Revenue":      (("rev_ex_duty", 1),),
    "Revenue duty": (("rev_duty", 1),),
    "COGS":         (("cogs_total", -1), ("cogs_duty", 1)),
    "COGS duty":    (("cogs_duty", -1),),
}


def pl_budget_amounts(country, year):
    """The four stream budgets off one P&L, in the units the rest of the workbook uses.

    The same four lines the budget sheet pulls out in Excel, worked out here in pandas so
    the net chart knows which of its labels will sit over a bar too short to carry one.
    Both sides read PL_TO_BUDGET, so the two cannot drift apart.
    """
    figures = pl_amounts(country, year)
    lines = {
        "rev_ex_duty": [a + b for a, b in zip(figures["rev_courier"], figures["rev_ff"])],
        "rev_duty": figures["rev_duty"],
        "cogs_total": [a + b + c for a, b, c in zip(figures["cogs_courier"],
                                                    figures["cogs_ff"],
                                                    figures["cogs_duty"])],
        "cogs_duty": figures["cogs_duty"],
    }
    amounts = {stream: [sum(sign * lines[key][month] for key, sign in terms) * PL_UNIT_SCALE
                        for month in range(len(PL_MONTHS))]
               for stream, terms in PL_TO_BUDGET.items()}
    # The scheduled lines need no rule of their own: each is one input row on the P&L,
    # entered negative like every other cost, and turned positive here the way the four
    # above already are.
    amounts.update({name: [-figure * PL_UNIT_SCALE for figure in figures[key]]
                    for key, name in SCHEDULED_LINES.items()})
    return amounts


def pl_budget_formula(row, columns):
    """One budget row's amount, read off the P&L for that country and year.

    Nested IFs on the stream rather than a different formula per row, so an amount follows
    an edit: change the Year, Month, Country or Stream on a row and it re-points itself,
    the same way the Key column beside it already does. INDIRECT builds the sheet name out
    of the row - `'P&L DNK 2026'` - and ADDRESS turns the month number into that sheet's
    column, month 1 being column C.

    The cost of that is the one thing worth knowing about this formula. An address built
    out of text is not a reference, so it does not follow a row inserted by hand on a P&L
    sheet the way `='P&L DNK 2026'!C13` would - it keeps pointing at row 13 whatever ends
    up there. The layout is written by this notebook, so that is a trade worth making, but
    insert a row up there in Excel and every one of these comes back one line short.

    Anything the four branches do not recognise falls through to "", and multiplying that
    is an error the IFERROR catches - so an unrecognised stream, and a country and year
    with no P&L sheet behind them, both come out as zero rather than as #REF!.
    """
    year, month = column_of(columns, "Year"), column_of(columns, "Month")
    country, stream = column_of(columns, "Country"), column_of(columns, "Stream")
    sheet = f'"\'{PL_SHEET_PREFIX} "&${country}{row}&" "&${year}{row}&"\'!"'
    column = f"${month}{row}+{PL_FIRST_MONTH_COLUMN - 1}"

    def line(key):
        return f"INDIRECT({sheet}&ADDRESS({PL_ROW_AT[key]},{column}))"

    branches = "".join(
        f'IF(${stream}{row}="{name}",'
        + "".join(f"{'+' if sign > 0 else '-'}{line(key)}" for key, sign in terms).lstrip("+")
        + ","
        for name, terms in PL_TO_BUDGET.items())
    # Everything else is a single input row read straight off a lookup, so the branches
    # stay four rather than sixteen and adding a cost line is a row on a table instead of
    # another level of nesting on two thousand formulas.
    fallback = (f"-INDIRECT({sheet}&ADDRESS("
                f"SUMIFS({COST_LINE_TABLE}[Row],{COST_LINE_TABLE}[Line],${stream}{row}),"
                f"{column}))")
    return f'=IFERROR(({branches}{fallback}{")" * len(PL_TO_BUDGET)})*{PL_UNIT_SCALE},0)'


# --- the sheet ------------------------------------------------------------------------
def write_profit_and_loss(book, country, year):
    """One country's P&L for one year: twelve months across, the year on the end."""
    figures = pl_amounts(country, year)
    sheet = new_sheet(book, pl_sheet_name(country, year), PL_WIDTHS)
    write_title(sheet, PL_TITLE)

    first = PL_FIRST_MONTH_COLUMN
    months = [get_column_letter(first + offset) for offset in range(len(PL_MONTHS))]
    total_letter = get_column_letter(PL_TOTAL_COLUMN)
    # Only the first year has anything closed on it, and the same rule sets the
    # Actuals/Budget column on the budget sheet.
    closed = FIRST_BUDGET_MONTH - 1 if year == BUDGET_YEARS[0] else 0

    # The year, once, over the months it covers.
    sheet.merge_cells(start_row=PL_YEAR_ROW, start_column=first,
                      end_row=PL_YEAR_ROW, end_column=first + len(PL_MONTHS) - 1)
    stamp = sheet.cell(row=PL_YEAR_ROW, column=first, value=year)
    stamp.font, stamp.number_format = YEAR_FONT, "0"
    stamp.alignment = Alignment(horizontal="center")

    for offset, month in enumerate(PL_MONTHS):
        basis = sheet.cell(row=PL_BASIS_ROW, column=first + offset,
                           value="Actual" if offset < closed else "Budget")
        basis.font = NOTE
        basis.alignment = Alignment(horizontal="center")
        head = sheet.cell(row=PL_HEADER_ROW, column=first + offset, value=month)
        head.font, head.border = HEADER, UNDERLINE
        head.alignment = Alignment(horizontal="center")

    # The unit sits where a heading for the label column would, and carries the currency
    # with it. The year column is headed on the far side of the hidden spacer.
    unit = sheet.cell(row=PL_HEADER_ROW, column=PL_LABEL_COLUMN, value=PL_UNIT)
    unit.font, unit.border = HEADER, UNDERLINE
    total = sheet.cell(row=PL_HEADER_ROW, column=PL_TOTAL_COLUMN, value="Total")
    total.font, total.border = HEADER, UNDERLINE
    total.alignment = Alignment(horizontal="center")
    sheet.column_dimensions[get_column_letter(PL_SPACER_COLUMN)].hidden = True

    for offset, (key, text, kind, rule) in enumerate(PL_ROWS):
        row = PL_FIRST_ROW + offset
        if kind in ("unused", "gap") and not PL_SHOW_UNUSED:
            sheet.row_dimensions[row].hidden = True
        if kind in ("blank", "gap"):
            continue                       # a spacer in the layout, left as one

        name = sheet.cell(row=row, column=PL_LABEL_COLUMN, value=text)
        name.font = HEADER if kind in ("title", "total") else BODY
        if kind == "title":
            continue                       # a heading carries no figures

        typed = kind in ("input", "unused")
        money = kind != "ratio"
        number_format = PL_MONEY if money else PL_PERCENT
        for position, letter in enumerate(months):
            cell = sheet[f"{letter}{row}"]
            if typed:
                if kind == "input":        # `unused` is typed too, and stays empty
                    cell.value = figures[key][position]
                cell.font = ENTRY          # blue, the convention for a figure that is keyed in
            else:
                cell.value = pl_sum(letter, rule) if money else pl_ratio(letter, rule)
                cell.font = HEADER if kind == "total" else BODY
            cell.number_format = number_format
            cell.alignment = Alignment(horizontal="right")
            if kind == "total":
                cell.border = TOP_RULE

        # The year. An amount adds across the twelve months, including the typed rows,
        # which is the one place a typed row carries a formula. A ratio is worked out from
        # the year's own totals rather than averaged over the months, because an average of
        # twelve margins is not the margin on the year.
        cell = sheet[f"{total_letter}{row}"]
        cell.value = (f"=SUM({months[0]}{row}:{months[-1]}{row})" if money
                      else pl_ratio(total_letter, rule))
        cell.font = HEADER if kind == "total" else BODY
        cell.number_format = number_format
        cell.alignment = Alignment(horizontal="right")
        if kind == "total":
            cell.border = TOP_RULE

    # The labels and the whole header block stay put while the months scroll.
    sheet.freeze_panes = f"{months[0]}{PL_FIRST_ROW}"
    return sheet

### The model assumptions sheet

Everything the forecast rests on that is *not* the budget: the calendars, the terms, the
mixes, the delays, and the handful of rules about how duty and settlement behave. All of
it lives in the notebook, which is fine for running the model and no use at all to
somebody holding the workbook and asking what is behind a number.

**It is read out of the objects themselves**, not typed out again here. `PAYMENT_TERMS`,
`CUSTOMER_MIX`, `PROVIDERS`, `PROVIDER_MIX`, `DELAY_CUSTOMER`, `DELAY_PROVIDER` and
`HOLIDAYS` are all walked and flattened. Change a mix in section 5 and this sheet changes
with it - it cannot end up describing a model other than the one that just ran.

Long, like the others: `Section`, `Assumption`, `Country`, `Value`, `Unit`, `Note`, one row
per fact. A term mix is forty-eight rows rather than a twelve-by-four block, which reads
worse and filters, sorts and pivots better - the same trade the three result sheets make.

The term mixes come out **normalised**, because that is what the model actually runs on;
the weights in the notebook do not have to add up to anything and are rescaled before use.

**It is a record, not an input.** Nothing reads it, and editing a cell on it changes
nothing - the note under the title says so, because a sheet full of assumptions is exactly
the sheet somebody will try to change. The values are in the notebook; change them there
and re-run.

**One thing it makes obvious.** `HOLIDAYS` covers 2026 and the horizon runs to the end of
2027, so every 2027 month currently loses weekends and nothing else. The sheet says which
years each country has a calendar for, which is how that shows up rather than staying
buried in a dict.

In [ ]:
ASSUMPTION_SHEET = "Model assumptions"
#                     header,   width, number format, alignment
ASSUMPTION_COLUMNS = [
    ("Section",           22, "@",       "left"),
    ("Assumption",        26, "@",       "left"),
    ("Country",           10, "@",       "center"),
    ("Value",             16, "General", "left"),
    ("Unit",               8, "@",       "left"),
    ("Note",              70, "@",       "left"),
]


def held_for(figures):
    """One entry per stretch of months a per-month setting keeps the same value over.

    (first month, last month, value), in horizon order. It is here for the sheet below
    rather than for the model: a figure set per country and month is ninety-six of them,
    and giving each its own row would bury the handful somebody actually changed under
    six hundred that they did not. A figure left alone throughout is one entry.
    """
    runs = []
    for period in periods:
        value = figures[month_key(period)]
        if runs and runs[-1][2] == value:
            runs[-1][1] = period
        else:
            runs.append([period, period, value])
    return runs


def over_months(first, last, note=""):
    """`note`, said of the months it holds over - and said plain where that is all of them.

    So a setting nobody has varied reads exactly as it did when there was only one of it,
    which is what most of them still are, and only one that changes has to name its months.
    """
    if first == periods[0] and last == periods[-1]:
        return note
    span = month_key(first) if first == last else f"{month_key(first)} to {month_key(last)}"
    return f"{span}: {note}" if note else span


def assumption_rows():
    """Every assumption the model runs on, one to a row.

    Read out of the objects themselves rather than typed out again here, so the sheet
    cannot describe a model other than the one that just ran. Change a mix in section 5
    and this sheet changes with it.
    """
    rows = []

    def add(section, assumption, country="", value="", unit="", note=""):
        rows.append((section, assumption, country, value, unit, note))

    # --- scope
    add("Scope", "First invoice month", value=START)
    add("Scope", "Horizon", value=HORIZON_MONTHS, unit="months",
        note=f"{START} to {periods[-1]:%Y-%m}")
    add("Scope", "Budget years", value=", ".join(str(year) for year in BUDGET_YEARS),
        note="one P&L per country per year")
    for country in COUNTRIES:
        add("Scope", "Country in scope", label(country), country)

    # --- invoicing
    add("Invoicing", "Spread within a month", value="equal",
        note="every work day carries 1 / number of work days - a placeholder for a real profile")
    add("Invoicing", "Work day", value="Mon-Fri",
        note="minus that country's bank holidays; invoicing only ever happens on one")

    for country in COUNTRIES:
        dates = HOLIDAYS[country]
        covered = {date.year for date in dates}
        blank = [str(year) for year in BUDGET_YEARS if year not in covered]
        add("Work-day calendar", "Bank holidays", label(country), len(dates), "days",
            f"best-effort national rules, generated for {min(covered)}-{max(covered)}" + (
                f"; nothing for {', '.join(blank)}, so those months lose weekends only"
                if blank else ""))
    for country in COUNTRIES:
        for date in HOLIDAYS[country]:
            add("Bank holidays", f"{date:%b}", label(country), f"{date:%Y-%m-%d}",
                note=date.strftime("%A"))

    # --- terms
    anchored = {"invoice": "counted from the invoice date",
                "month_end": "counted from the last day of the invoice month"}
    for term, (anchor, days) in PAYMENT_TERMS.items():
        add("Payment terms", term, value=days, unit="days", note=anchored[anchor])

    # Normalised, because that is what the model runs on - the weights themselves do not
    # have to add up to anything.
    shares = CUSTOMER_MIX / CUSTOMER_MIX.sum() * 100
    for term in CUSTOMER_MIX.index:
        for country in COUNTRIES:
            add("Customer term mix", term, label(country),
                round(float(shares.at[term, country]), 1), "%",
                "placeholder - share of revenue invoiced on this term")

    for provider, term in PROVIDERS.items():
        add("Providers", provider, value=term,
            note="the term this provider invoices us on - the same in every country")

    shares = PROVIDER_MIX / PROVIDER_MIX.sum() * 100
    for provider in PROVIDER_MIX.index:
        for country in COUNTRIES:
            add("Provider mix", provider, label(country),
                round(float(shares.at[provider, country]), 1), "%",
                "placeholder - share of COGS spend going to this provider")

    # --- delays
    for section, table, note in (
            ("Customer payment delay", DELAY_CUSTOMER,
             "placeholder - days late against the due date"),
            ("Provider payment delay", DELAY_PROVIDER,
             "placeholder - we assume we pay on the day it falls due")):
        for term in table.index:
            for country in COUNTRIES:
                add(section, term, label(country), int(table.at[term, country]), "days", note)

    # --- streams and duty
    for stream in STREAMS_IN:
        add("Streams", STREAM_LABELS[stream], value="money in")
    for stream in STREAMS_OUT:
        add("Streams", STREAM_LABELS[stream], value="money out")
    add("Duty", "How it is budgeted", value="per side",
        note="budgeted in its own right on each side, never a rate applied to revenue")
    add("Duty", "Term mix", value="its service stream's",
        note="duty rides the same invoice, so it falls due on the same day")
    add("Duty", "Payment delay", value="its service stream's",
        note="duty runs as late as whatever it is invoiced alongside")

    # --- settlement
    add("Settlement", "Due dates", value="nominal days",
        note="a due date can and does land on a weekend or a bank holiday")
    add("Settlement", "Payment dates", value="rolled forward",
        note="to the next work day on that country's own calendar")

    # --- factoring
    # 1 January 2024 was a Monday, which is what turns a weekday number into its name.
    run = pd.Timestamp("2024-01-01") + pd.Timedelta(days=INVOICE_RUN_WEEKDAY)
    add("Factoring", "Invoice run", value=f"{run:%A}",
        note="the week's factored invoices go out on the first one strictly after the work")
    add("Factoring", "Advance lands", value=f"{run + pd.Timedelta(days=ADVANCE_OFFSET_DAYS):%A}",
        note=f"{ADVANCE_OFFSET_DAYS} days on, counted from the nominal run day rather than "
             "a rolled one, then rolled itself onto a work day")
    add("Factoring", "Bank delay", value=BANK_DELAY_DAYS, unit="days",
        note="what the bank takes to pass the retention on once the customer has paid it")
    add("Factoring", "Streams factored",
        value=", ".join(STREAM_LABELS[stream] for stream in FACTORED_STREAMS),
        note="the bank buys the invoice, and duty is a line on it, so it is sold with it")
    add("Factoring", "Term mix of the factored slice", value="the country's own",
        note="there is no customer dimension, so the factored share sits on the same terms "
             "as everyone else")
    add("Factoring", "Facility on or off",
        value="notebook" if FACTORING_IS_A_NOTEBOOK_SETTING else "workbook",
        note=("set here, so a country that does not factor carries no factoring rows at all"
              if FACTORING_IS_A_NOTEBOOK_SETTING else
              "every country carries the legs whether it uses them or not, and the Yes/No "
              "on the Factoring sheet is what empties them"))
    add("Factoring", "Rates", value="Factoring sheet",
        note="typed in the workbook - the figures below are what it was built with")

    # All four of these are set per country and per month, so each is read out a stretch
    # at a time: one row for the whole horizon where nobody has varied it, and a row per
    # stretch where somebody has, each naming the months it covers.
    for country in COUNTRIES:
        for first, last, on in held_for(FACTORING[country]):
            add("Factoring", "Facility", label(country), "Yes" if on else "No",
                note=over_months(first, last))
    for what, config, note in (
            ("Factored share", FACTORED_SHARE,
             "share of that country's invoicing the bank buys"),
            ("Advance rate", ADVANCE_RATE,
             "what the bank pays out on the day it buys an invoice"),
            ("Fee", FACTORING_FEE,
             "the bank's cut, as a share of the invoice face value and not of the retention")):
        for country in COUNTRIES:
            for first, last, rate in held_for(config[country]):
                add("Factoring", what, label(country), round(rate * 100, 2), "%",
                    over_months(first, last, note))

    # What that comes to per leg. The four add to one invoice, which is what keeps the
    # cash reconciling to the budget with the fee carried as a leg of its own.
    for country in COUNTRIES:
        for leg, name in LEG_LABELS.items():
            rates = pd.Series({month_key(period): leg_rates(country, period, "revenue")[leg]
                               for period in periods})
            for first, last, rate in held_for(rates):
                add("Factoring legs", name, label(country), round(rate * 100, 3), "%",
                    over_months(first, last,
                                "share of a revenue invoice reaching cash this way"))

    # --- cost timing
    explained = {
        "month_end": "all of it on the last working day of the month",
        "mid_month": f"all of it on the {MID_MONTH_DAY}th, or the next working day",
        "spread":    "equal across the month's working days",
        "week":      "all of it in one week of the month - nothing uses this yet",
        "none":      "written off rather than paid out, so it never reaches cash",
    }
    for rule, name in TIMING_RULES.items():
        add("Cost timing rules", name, note=explained[rule])
    add("Cost timing", "Where it is set", value="Cost timing sheet",
        note="a dropdown per country and line - the figures below are what it was built with")
    add("Cost timing", "Grain", value="weeks",
        note="a month is cut into the weeks its working days touch, and every rule is "
             "arithmetic over five facts about each")
    for line, name in SCHEDULED_LINES.items():
        for country in COUNTRIES:
            add("Cost timing", name, label(country),
                TIMING_RULES[COST_TIMING.at[line, country]])

    # --- cash flow
    add("Cash flow", "Years", value=", ".join(str(year) for year in CASH_YEARS),
        note="one sheet per country per year; the last runs past the budget because the "
             "tail of the horizon is still cash")
    add("Cash flow", "Columns", value="ISO weeks",
        note="grouped under their month, and collapsing a group leaves that month's total")
    add("Cash flow", "Weeks across a month end", value="split",
        note="a week straddling two months is a column under each, holding only that "
             "month's days, so a month total is a real calendar month")
    add("Cash flow", "Unit", value="thousands", unit=PL_UNIT,
        note="a number format only - the cells hold whole units and tie to Payment date cash")

    # --- budget
    add("Budget", "Reporting currency", value="DKK",
        note="every P&L is in DKK, whichever country it is for")
    add("Budget", "P&L unit", value="thousands", unit="DKKt",
        note=f"amounts everywhere else are whole units - the budget sheet multiplies by {PL_UNIT_SCALE:,}")
    add("Budget", "Actuals through", value=f"{BUDGET_YEARS[0]}-{FIRST_BUDGET_MONTH - 1:02d}",
        note="every month after this one is a budget")
    add("Budget", "Figures on the P&L sheets", value="random placeholders",
        note="seeded per country and year - replace them with the real budget")
    if BUDGET_ZERO_YEARS:
        add("Budget", "Years left empty",
            value=", ".join(str(year) for year in BUDGET_ZERO_YEARS),
            note="the sheets and their formulas are there; nothing is typed on them yet")
    return rows


def write_model_assumptions(book):
    """What the model assumes, written out of the objects it assumes them in."""
    sheet = new_sheet(book, ASSUMPTION_SHEET, [width for _, width, _, _ in ASSUMPTION_COLUMNS])
    write_title(sheet, ASSUMPTION_SHEET)
    write_note(sheet, "This sheet shows the assumptions and variables that the python "
                      "script used to run")
    write_table(sheet, "ModelAssumptions", ASSUMPTION_COLUMNS, assumption_rows())
    return sheet

### The budget sheet

Everything above is a percentage, which is what let the model be built without any
amounts at all. This is where the amounts come in, to be multiplied through the shares.

**Four streams, four budgets.** The model runs the service and the duty on it as separate
decompositions on each side, each a share of *its own* monthly budget, so it needs four
figures per country and month:

| stream | is | read off the P&L as |
|---|---|---|
| `Revenue` | the service, excluding duty | `Revenue total ex. duty` |
| `Revenue duty` | the duty recharged to the customer | `Duty`, under Revenue |
| `COGS` | freight, courier and forwarding, excluding duty | `Cost of Goods sold` less its `Duty` |
| `COGS duty` | the duty paid on the import | `Duty`, under Cost of Goods sold |

`Revenue Total` and `COGS Total` are then sums of their pair - which is why they are not
columns here. Storing a total beside its parts means one of the three can disagree with
the other two, and a long table gets its totals from a pivot for free.

That shape matches the three result sheets exactly: `Key` plus `Stream`, long rather than
wide. Applying the amounts is a join on those two columns rather than a reshape.

### It is not typed on any more

`Amount` used to be a column of figures keyed in here. It is now a **formula reading the
P&L** for that row's country and year, which is the whole point of the exercise: the
budget is kept once, on the human-readable sheet, and this one only reshapes it into the
long form the model wants.

```
=IFERROR((IF($F5="Revenue",      INDIRECT("'P&L "&$E5&" "&$C5&"'!"&ADDRESS(13,$D5+2)),
          IF($F5="Revenue duty", INDIRECT(... ADDRESS(11, ...)),
          IF($F5="COGS",        -INDIRECT(... ADDRESS(19, ...))+INDIRECT(... ADDRESS(18, ...)),
          IF($F5="COGS duty",   -INDIRECT(... ADDRESS(18, ...)),
          "")))))*1000, 0)
```

Three things are going on in there.

**It follows an edit.** Nested `IF`s on the stream rather than a different formula per
row, so changing the Year, Month, Country or Stream on a row re-points it - the same way
the `Key` column beside it already does. `INDIRECT` builds the sheet name out of the row
itself, `'P&L DNK 2026'`, and `ADDRESS` turns the month number into that sheet's column,
month 1 being column C.

**The signs flip.** Costs are negative on a P&L and the budget holds positive magnitudes,
because the sign that says *money out* is applied downstream by the charts and the net
column. So the two cost streams come across negated. `COGS` is the cost line with its duty
taken back out, because duty is a stream in its own right and would otherwise be counted
twice - the `19-18` in the formula.

**The scale changes.** The P&L is in thousands and everything else here is in whole units,
which is the `*1000`. That is also why the net chart's `k` formatting still reads right.

**What it costs.** An address built out of text is not a reference, so it does not follow
a row inserted by hand on a P&L sheet the way `='P&L DNK 2026'!C13` would - it keeps
pointing at row 13 whatever ends up there. The P&L layout is written by this notebook, so
that is a trade worth making. But insert a row up there in Excel and every one of these
comes back one line short. The `IFERROR` turns an unrecognised stream, or a country and
year with no P&L behind them, into a zero rather than a `#REF!`.

`Key` is `Year-Month-Country`, also a **formula** rather than a typed value, so it follows
an edited row and a row added to the table gets one without anybody having to remember.
Month is zero-padded inside the key so keys sort in date order, even though the `Month`
column itself is a plain 1-12. Stream stays out of the key and on its own column, exactly
as it is on the result sheets.

Month, Country, Stream and Actuals/Budget are all dropdowns, which is why the sheet needs
no legend: the valid values are in the cells rather than written out above them. The four
dimensions are the only blue on the sheet now - `Key` and `Amount` are both black, because
neither is typed here.

It arrives covering the whole horizon: both years, all twelve months, four countries and
four streams, 384 rows, actuals to April of the first year and budget from May.

**On formulas and blank cells.** openpyxl writes a formula as text with no cached result,
so a workbook it has just written has nothing for Excel to show until something
recalculates it - which is why the key column came out empty the first time. Setting
`fullCalcOnLoad` makes Excel recalculate the whole workbook as it opens the file, so every
formula in here fills itself in on the way in. Worth knowing if you ever read this file
with pandas *without* opening it in Excel first: the formula columns come back as `NaN`,
because there is still nothing cached for them - and that now includes every amount.

In [ ]:
def placeholder_budget():
    """A row per country, year, month and stream, and what the P&L behind it is worth.

    Key and Amount are both left to the sheet's own formulas. The amounts worked out here
    are the same four lines in pandas, and exist only so the net chart can tell which of
    its labels would sit over a bar too short to carry one.

    The whole horizon is written out, closed months included. A missing row and a zero row
    look identical on a chart and nothing like each other in a lookup: the row that is not
    there returns nothing and drops silently out of the model, while the zero says the
    month is known about and waiting to be filled in.
    """
    rows = []
    for year in BUDGET_YEARS:
        for country in COUNTRIES:
            amounts = pl_budget_amounts(country, year)
            for month in range(1, 13):
                # The same rule that sets Actual or Budget across the top of each P&L.
                settled = year == BUDGET_YEARS[0] and month < FIRST_BUDGET_MONTH
                for stream in CATEGORY_LABELS.values():
                    rows.append((None, year, month, label(country), stream,
                                 amounts[stream][month - 1],
                                 "Actuals" if settled else "Budget"))
    return rows


def budget_lookup():
    """The amounts, keyed the way the sheets are.

    Only used to work out what the net chart will hold, so its labels can be thinned. The
    sheet itself always looks the amount up in Excel.
    """
    return {(f"{year}-{month:02d}-{country}", stream): amount
            for _, year, month, country, stream, amount, _ in placeholder_budget()}


# --- the sheet ----------------------------------------------------------------------
# Long, and in the same column order as the result sheets, so applying the amounts is a
# join on Key and Stream rather than a reshape first.
BUDGET_TABLE = "BudgetInput"   # the three result sheets look their amounts up in this table
#                 header,        width, number format, alignment
BUDGET_COLUMNS = [
    ("Key",            18, "@",     "left"),
    ("Year",            8, "0",     "center"),
    ("Month",           8, "0",     "center"),
    ("Country",        10, "@",     "center"),
    ("Stream",         24, "@",     "left"),
    ("Amount",         14, "#,##0", "right"),
    ("Actuals/Budget", 16, "@",     "left"),
]
SPARE_ROWS = 100           # how far past the data the dropdowns reach, for rows added by hand


def write_budget_input(book):
    """The bridge sheet: one row per country, month and stream, and what it is worth."""
    rows = placeholder_budget()
    sheet = new_sheet(book, "Budget input", [width for _, width, _, _ in BUDGET_COLUMNS])
    write_title(sheet, "Budget input")
    # Blue, because the four dimensions are picked here. Key and Amount are set back to
    # black below - both are formulas, and neither is typed on this sheet any more.
    write_table(sheet, BUDGET_TABLE, BUDGET_COLUMNS, rows, font=ENTRY)

    key, year = column_of(BUDGET_COLUMNS, "Key"), column_of(BUDGET_COLUMNS, "Year")
    month, country = column_of(BUDGET_COLUMNS, "Month"), column_of(BUDGET_COLUMNS, "Country")
    stream, kind = column_of(BUDGET_COLUMNS, "Stream"), column_of(BUDGET_COLUMNS, "Actuals/Budget")
    amount = column_of(BUDGET_COLUMNS, "Amount")

    last_row = FIRST_DATA_ROW + len(rows) - 1
    for row in range(FIRST_DATA_ROW, last_row + 1):
        cell = sheet[f"{key}{row}"]
        # Zero-padded month inside the key so keys sort in date order; the Month column
        # itself stays a plain 1-12. Stream is left out of the key and stays its own
        # column, exactly as it is on the result sheets.
        cell.value = f'={year}{row}&"-"&TEXT({month}{row},"00")&"-"&{country}{row}'
        cell.font = BODY   # a formula, not an input

        cell = sheet[f"{amount}{row}"]
        # Read off the P&L for this row's country and year rather than typed here, so the
        # budget is kept in one place and this sheet only reshapes it.
        cell.value = pl_budget_formula(row, BUDGET_COLUMNS)
        cell.font = BODY

    add_dropdown(sheet, month, range(1, 13), FIRST_DATA_ROW, last_row + SPARE_ROWS)
    add_dropdown(sheet, country, COUNTRY_LABELS.values(), FIRST_DATA_ROW, last_row + SPARE_ROWS)
    add_dropdown(sheet, stream, CATEGORY_LABELS.values(), FIRST_DATA_ROW, last_row + SPARE_ROWS)
    add_dropdown(sheet, kind, ("Actuals", "Budget"), FIRST_DATA_ROW, last_row + SPARE_ROWS)
    return sheet

### The factoring sheet

Two blocks side by side, sharing a header row. On the left is what the facility says in one
country in one month - how much of that month's invoicing the bank buys, what it advances
against it, and what it charges. Those are typed, and they are the numbers worth arguing
about with a bank. Four countries over twenty-four months is ninety-six rows of them, and
changing one changes that month alone. On the right is what they work out to per leg: one
row for every country, month, stream and leg the notebook wrote, with a rate against each.
Every rate is a formula reading the block on the left, so changing one advance from 90% to
85% moves that month's rows on both cash sheets without this notebook running again.

Both blocks carry `Key`, the year, month and country in one string, which is the same key
the budget sheet and the cash sheets are built on. The rate rows carry `Rate key` as well -
that key with the stream and the leg on the end - because it is the single thing a cash row
matches on. Building it as a column costs a thousand formulas here and saves four SUMIFS
conditions on every one of tens of thousands of cash rows.

The right-hand block is generated from the same rule the legs themselves are, so a lookup
coming from a cash sheet can never fail to find its row. Effective share is what makes the
Factoring column load-bearing: it is nought whenever the answer there is No, which hands
the whole invoice back to the direct leg and leaves anything typed beside it inert - one
month at a time, so a facility that lapses for a quarter is three Nos and nothing else.

In [ ]:
FACTORING_SHEET = "Factoring"
FACTORING_TABLE = "FactoringInput"
RATE_TABLE = "FactoringRates"

#                     header,        width, number format, alignment
# Year, Month and Country are the three dimensions the facility is set on, and Key is the
# three of them in one string - the same key the budget sheet and the two cash sheets are
# built on, so one table can be found from another on a single criterion.
FACTORING_COLUMNS = [
    ("Key",               14, "@",     "left"),
    ("Year",               8, "0",     "center"),
    ("Month",              8, "0",     "center"),
    ("Country",           10, "@",     "center"),
    ("Factoring",         11, "@",     "center"),
    ("Factored share",    15, "0.0%",  "right"),
    ("Advance rate",      14, "0.0%",  "right"),
    ("Fee rate",          11, "0.00%", "right"),
    ("Effective share",   16, "0.0%",  "right"),
]
RATE_COLUMNS = [
    # Key with the stream and the leg on the end: everything it takes to tell one rate
    # from another, in the one column the cash sheets match on.
    ("Rate key",          46, "@",      "left"),
    ("Key",               14, "@",      "left"),
    ("Year",               8, "0",      "center"),
    ("Month",              8, "0",      "center"),
    ("Country",           10, "@",      "center"),
    ("Stream",            14, "@",      "left"),
    ("Leg",               19, "@",      "left"),
    ("Rate",              12, "0.000%", "right"),
]
RATE_FIRST_COLUMN = 2 + len(FACTORING_COLUMNS) + 1     # a blank column between the blocks


def factoring_legs():
    """Every country, month, stream and leg the notebook wrote a row for.

    Generated from the same rule apply_factoring() runs on, so the rate table covers the
    cash sheets exactly - no row on them can look up a rate that is not here, and no rate
    here sits unread.

    Every month of the horizon, whatever the facility says in each of them, for the same
    reason factors_ever() decides the country: the rows are fixed when this notebook runs
    and the rate is what varies. A month answering No gets its four legs with nought on
    three of them, which is the whole invoice back on the direct leg.
    """
    for country in COUNTRIES:
        factored = factors_ever(country) or not FACTORING_IS_A_NOTEBOOK_SETTING
        for period in periods:
            for stream in STREAM_LABELS:
                legs = LEG_LABELS if stream in FACTORED_STREAMS and factored else ("direct",)
                for leg in legs:
                    yield country, period, stream, leg


def write_factoring(book):
    """What the facility costs in each country and month, and the rate that puts on each leg."""
    gap = [MARGIN_WIDTH]
    widths = ([width for _, width, _, _ in FACTORING_COLUMNS] + gap
              + [width for _, width, _, _ in RATE_COLUMNS])
    sheet = new_sheet(book, FACTORING_SHEET, widths)
    write_title(sheet, FACTORING_SHEET)

    inputs = [(None,                                  # Key, a formula below
               period.year, period.month, label(country),
               "Yes" if FACTORING.at[month_key(period), country] else "No",
               FACTORED_SHARE.at[month_key(period), country],
               ADVANCE_RATE.at[month_key(period), country],
               FACTORING_FEE.at[month_key(period), country],
               None)                                  # Effective share, a formula below
              for country in COUNTRIES for period in periods]
    written = write_table(sheet, FACTORING_TABLE, FACTORING_COLUMNS, inputs, font=ENTRY)

    key = column_of(FACTORING_COLUMNS, "Key")
    year, month = column_of(FACTORING_COLUMNS, "Year"), column_of(FACTORING_COLUMNS, "Month")
    country_column = column_of(FACTORING_COLUMNS, "Country")
    switch = column_of(FACTORING_COLUMNS, "Factoring")
    share = column_of(FACTORING_COLUMNS, "Factored share")
    effective = column_of(FACTORING_COLUMNS, "Effective share")
    last_row = FIRST_DATA_ROW + written - 1
    for row in range(FIRST_DATA_ROW, last_row + 1):
        cell = sheet[f"{key}{row}"]
        # Zero-padded month inside the key so keys sort in date order, exactly as the
        # budget sheet builds its own; the Month column itself stays a plain 1-12.
        cell.value = f'={year}{row}&"-"&TEXT({month}{row},"00")&"-"&{country_column}{row}'
        cell.font = BODY                              # a formula, not an input

        cell = sheet[f"{effective}{row}"]
        # Nought when the answer is No, which is what makes turning a month off safe:
        # the direct leg goes back to the whole invoice and nothing typed beside it counts.
        cell.value = f'=IF(${switch}{row}="Yes",${share}{row},0)'
        cell.font = BODY                              # worked out, not typed
    if FACTORING_IS_A_NOTEBOOK_SETTING:
        # Set in the notebook, so it is shown the way every other derived figure is shown
        # and not in the blue that means type here.
        for row in range(FIRST_DATA_ROW, last_row + 1):
            sheet[f"{switch}{row}"].font = BODY
    else:
        add_dropdown(sheet, switch, ("Yes", "No"), FIRST_DATA_ROW, last_row + SPARE_ROWS)

    legs = list(factoring_legs())
    rates = [(None,                                   # Rate key, a formula below
              None,                                   # Key, a formula below
              period.year, period.month, label(country),
              STREAM_LABELS[stream], LEG_LABELS[leg],
              None)                                   # Rate, a formula below
             for country, period, stream, leg in legs]
    write_table(sheet, RATE_TABLE, RATE_COLUMNS, rates, first_column=RATE_FIRST_COLUMN)

    def rate_letter(header):
        return column_of(RATE_COLUMNS, header, RATE_FIRST_COLUMN)

    rate_key, row_key = rate_letter("Rate key"), rate_letter("Key")
    rate_year, rate_month = rate_letter("Year"), rate_letter("Month")
    rate_country, rate_stream = rate_letter("Country"), rate_letter("Stream")
    rate_leg, rate_column = rate_letter("Leg"), rate_letter("Rate")

    def parameter(header, row):
        """One of the typed figures, picked off the block on the left by month and country."""
        return (f"SUMIFS({FACTORING_TABLE}[{header}],{FACTORING_TABLE}[Key],"
                f"${row_key}{row})")

    for offset, (_, _, stream, leg) in enumerate(legs):
        row = FIRST_DATA_ROW + offset
        sheet[f"{row_key}{row}"] = (
            f'={rate_year}{row}&"-"&TEXT({rate_month}{row},"00")&"-"&{rate_country}{row}')
        # The one thing a cash sheet matches on. Built here rather than asked for as three
        # SUMIFS conditions because those sheets have tens of thousands of rows and each of
        # them puts this question to this table once.
        sheet[f"{rate_key}{row}"] = (f'=${row_key}{row}&"|"&${rate_stream}{row}'
                                     f'&"|"&${rate_leg}{row}')

        factored = parameter("Effective share", row)
        advance, fee = parameter("Advance rate", row), parameter("Fee rate", row)
        if stream not in FACTORED_STREAMS:
            formula = "=1"                            # nothing on the COGS side is sold
        elif leg == "direct":
            formula = f"=1-{factored}"
        elif leg == "advance":
            formula = f"={factored}*{advance}"
        elif leg == "retention":
            # What is left of the held-back slice once the bank has taken its fee out.
            formula = f"={factored}*(1-{advance}-{fee})"
        else:
            formula = f"={factored}*{fee}"
        sheet[f"{rate_column}{row}"] = formula
    return sheet


### The cost timing sheet, and the costs it dates

Cost timing is where the rules from section 11 are set, one row per country and cost line,
each with a dropdown carrying the five of them. Change one and the cash moves, without this
notebook running again. Beside it sits the row each cost line is budgeted on, which is what
lets the budget sheet read sixteen lines off a P&L instead of four without growing sixteen
nested IFs to do it.

Cost cash is the result: one row per country, cost line, month and week. Its share of the
month is a formula rather than a number - it reads the rule off Cost timing and the five
facts off the week block beside it, and works out the same share section 11 works out in
pandas. The two agree because they are the same arithmetic written twice against the same
facts.

The week block is the small one, four countries by twenty-four months by about five weeks,
and it does not depend on the cost line, so one copy serves all twelve. Three of the five
rules resolve to a column on it outright; the other two need the line's own setting, which
is why the share is worked out on Cost cash rather than there.

The budget sheet now carries all sixteen lines rather than four. Nothing else about it
changes: the amount is still read off the P&L for that row's country and year, and the
result sheets still look it up on Key and Stream.

In [ ]:
COST_TIMING_SHEET = "Cost timing"
TIMING_TABLE = "CostTiming"
COST_LINE_TABLE = "CostLines"     # the budget formula reads the P&L row numbers off this

missing = [line for line in SCHEDULED_LINES if line not in PL_ROW_AT]
if missing:
    raise KeyError(f"No P&L row for scheduled cost line(s): {missing}")

#                  header,        width, number format, alignment
TIMING_COLUMNS = [
    ("Key",           24, "@", "left"),
    ("Country",       10, "@", "center"),
    ("Cost line",     24, "@", "left"),
    ("Timing",        30, "@", "left"),
    ("Week",           8, "0", "center"),
]
COST_LINE_COLUMNS = [
    ("Line",          24, "@", "left"),
    ("Row",            8, "0", "center"),
]
COST_LINE_FIRST_COLUMN = 2 + len(TIMING_COLUMNS) + 1


def timing_key(country, name):
    """One country's setting for one cost line. Written both sides, so a lookup is exact."""
    return f"{label(country)}|{name}"


def write_cost_timing(book):
    """When each cost line pays, per country - and which P&L row each one is budgeted on."""
    widths = ([width for _, width, _, _ in TIMING_COLUMNS] + [MARGIN_WIDTH]
              + [width for _, width, _, _ in COST_LINE_COLUMNS])
    sheet = new_sheet(book, COST_TIMING_SHEET, widths)
    write_title(sheet, COST_TIMING_SHEET)
    write_note(sheet, "Set when each cost reaches cash. The block on the right is which "
                      "P&L row each line is budgeted on, and is not typed.")

    rows = [(timing_key(country, SCHEDULED_LINES[line]), label(country),
             SCHEDULED_LINES[line], TIMING_RULES[COST_TIMING.at[line, country]],
             int(COST_TIMING_WEEK.at[line, country]))
            for country in COUNTRIES for line in SCHEDULED_LINES]
    written = write_table(sheet, TIMING_TABLE, TIMING_COLUMNS, rows, font=ENTRY)

    last_row = FIRST_DATA_ROW + written - 1
    for header in ("Key", "Country", "Cost line"):
        letter = column_of(TIMING_COLUMNS, header)
        for row in range(FIRST_DATA_ROW, last_row + 1):
            sheet[f"{letter}{row}"].font = BODY      # named by the notebook, not picked here
    add_dropdown(sheet, column_of(TIMING_COLUMNS, "Timing"), TIMING_RULES.values(),
                 FIRST_DATA_ROW, last_row)
    add_dropdown(sheet, column_of(TIMING_COLUMNS, "Week"), range(1, 6),
                 FIRST_DATA_ROW, last_row)

    write_table(sheet, COST_LINE_TABLE, COST_LINE_COLUMNS,
                [(name, PL_ROW_AT[line]) for line, name in SCHEDULED_LINES.items()],
                first_column=COST_LINE_FIRST_COLUMN)
    return sheet


# --- the costs, dated ------------------------------------------------------------------
COST_CASH_SHEET = "Cost cash"
COST_CASH_TABLE = "CostCash"
WEEK_FACT_TABLE = "CostWeeks"

COST_CASH_COLUMNS = [
    ("Key",            18, "@",      "left"),
    ("Year",            8, "0",      "center"),
    ("Month",           8, "0",      "center"),
    ("Country",        10, "@",      "center"),
    ("Stream",         24, "@",      "left"),
    ("Leg",            10, "@",      "left"),
    ("Flow",            7, "@",      "center"),
    ("Cash week",      14, "@",      "center"),
    ("Cash key",       38, "@",      "left"),
    ("Week key",       11, "@",      "center"),
    ("Timing key",     24, "@",      "left"),
    ("Week lookup",    20, "@",      "left"),
    ("Timing",         30, "@",      "left"),
    ("Week",            8, "0",      "center"),
    ("Share of month", 15, "0.000%", "right"),
    ("Cash amount",    16, "#,##0",  "right"),
]
WEEK_FACT_COLUMNS = [
    ("Key",            20, "@",      "left"),
    ("Country",        10, "@",      "center"),
    ("Cash week",      14, "@",      "center"),
    ("Week key",       11, "@",      "center"),
    ("Week of month",  13, "0",      "center"),
    ("Weeks in month", 14, "0",      "center"),
    ("Days",            8, "0",      "center"),
    ("Month days",     11, "0",      "center"),
    ("Month end",      11, "0",      "center"),
    ("Mid month",      11, "0",      "center"),
    ("Spread",         11, "0.000%", "right"),
]
WEEK_FACT_FIRST_COLUMN = 2 + len(COST_CASH_COLUMNS) + 1


def cash_week_of(date):
    """The column one payment lands in: the month it falls in, and its ISO week.

    Keyed on the month of the date rather than the ISO week's own, so a week straddling
    two months is two columns and each month's total is a real calendar month.
    """
    return f"{date.year}-{date.month:02d}-W{date.isocalendar()[1]:02d}"


def cash_week_in(period, week_key):
    """The same key for a week of a month, where the day itself is not carried."""
    return f"{period.year}-{period.month:02d}-{week_key.split('-')[-1]}"


def week_fact_key(country, cash_week):
    """A country's own copy of one week: the calendars differ, so the facts do."""
    return f"{label(country)}|{cash_week}"


def cost_share_formula(row):
    """One week's share of a month's cost, worked out from the rule and the week facts.

    The same five rules section 11 runs in pandas, against the same five facts. Excel
    stops at the first branch that matches, so a month-end line never touches the spread
    column and the sheet is cheaper than the nesting makes it look.
    """
    rule = column_of(COST_CASH_COLUMNS, "Timing")
    week = column_of(COST_CASH_COLUMNS, "Week")
    key = column_of(COST_CASH_COLUMNS, "Week lookup")

    def fact(header):
        return (f"SUMIFS({WEEK_FACT_TABLE}[{header}],{WEEK_FACT_TABLE}[Key],"
                f"${key}{row})")

    # Clamped, so asking for the fifth week of a four-week month pays in the fourth
    # rather than paying not at all.
    chosen = f"MIN(${week}{row},{fact('Weeks in month')})"
    return (f'=IF(${rule}{row}="{TIMING_RULES["none"]}",0,'
            f'IF(${rule}{row}="{TIMING_RULES["month_end"]}",{fact("Month end")},'
            f'IF(${rule}{row}="{TIMING_RULES["mid_month"]}",{fact("Mid month")},'
            f'IF(${rule}{row}="{TIMING_RULES["spread"]}",{fact("Spread")},'
            f'IF(${rule}{row}="{TIMING_RULES["week"]}",'
            f'IF({fact("Week of month")}={chosen},1,0),0)))))')


def write_cost_cash(book):
    """Every scheduled cost, at the week grain the cash flow sheets read it in."""
    widths = ([width for _, width, _, _ in COST_CASH_COLUMNS] + [MARGIN_WIDTH]
              + [width for _, width, _, _ in WEEK_FACT_COLUMNS])
    sheet = new_sheet(book, COST_CASH_SHEET, widths)
    write_title(sheet, COST_CASH_SHEET)
    write_note(sheet, "One row per country, cost line, month and week. The share is read "
                      "off Cost timing and the week block on the right.")

    rows = []
    for cost in costs.itertuples():
        name = SCHEDULED_LINES[cost.line]
        cash_week = cash_week_in(cost.period, cost.week_key)
        rows.append((f"{cost.period.year}-{cost.period.month:02d}-{label(cost.country)}",
                     cost.period.year, cost.period.month, label(cost.country), name,
                     LEG_LABELS["direct"], "Out", cash_week,
                     f"{label(cost.country)}|{name}|{cash_week}", cost.week_key,
                     timing_key(cost.country, name),
                     week_fact_key(cost.country, cash_week),
                     None, None, None, None))     # the last four are formulas, below
    written = write_table(sheet, COST_CASH_TABLE, COST_CASH_COLUMNS, rows)

    timing = column_of(COST_CASH_COLUMNS, "Timing key")
    for row in range(FIRST_DATA_ROW, FIRST_DATA_ROW + written):
        for header, field in (("Timing", "Timing"), ("Week", "Week")):
            sheet[f"{column_of(COST_CASH_COLUMNS, header)}{row}"] = (
                f"=INDEX({TIMING_TABLE}[{field}],MATCH(${timing}{row},"
                f"{TIMING_TABLE}[Key],0))")
        sheet[f'{column_of(COST_CASH_COLUMNS, "Share of month")}{row}'] = (
            cost_share_formula(row))
    write_amounts(sheet, COST_CASH_COLUMNS, written, "Cash amount")

    facts = [(week_fact_key(week["country"], cash_week_in(week["period"], week["week_key"])),
              label(week["country"]),
              cash_week_in(week["period"], week["week_key"]), week["week_key"],
              week["week_of_month"], week["weeks_in_month"], week["days"],
              week["month_days"], week["has_last"], week["has_mid"],
              week["days"] / week["month_days"])
             for week in COST_WEEKS.to_dict("records")]
    write_table(sheet, WEEK_FACT_TABLE, WEEK_FACT_COLUMNS, facts,
                first_column=WEEK_FACT_FIRST_COLUMN)
    return sheet


### The dashboard

It reads the P&L sheets, and nothing on it is typed except the four dropdowns. Change a
figure on a P&L and this follows, which is the point of putting it here rather than keeping
a second copy of the budget.

The top block is one P&L at a time, chosen by a country and a year, with a second country
and year beside it and the difference between the two. Under it is a matrix: every country
in its own column for the selected year, and a group total.

Both country dropdowns carry All as well as the four countries, which adds the four P&Ls
up. That is applied to money only. A ratio row is worked out from its own two lines on the
dashboard itself rather than pulled off a sheet, because the group's margin is not the sum
of four margins - and the same rule is what makes the comparison and the total columns
right without a special case.

The sheet name each figure comes from is rebuilt in Excel out of the two dropdowns, the way
the budget sheet already rebuilds it. So `pl_sheet_name()` and the spaces inside it are
load-bearing in a second place now.

In [ ]:
ALL_COUNTRIES = "All"          # the extra option on both country dropdowns
DASHBOARD_SHEET = "Dashboard"

# The two selections, then a caption, the Actual/Budget basis pulled off the chosen sheet,
# and the month headers. The line rows start two below, so the block reads with a gap.
DASH_SELECT_ROW, DASH_COMPARE_ROW = 5, 6
DASH_CAPTION_ROW, DASH_BASIS_ROW, DASH_HEADER_ROW = 9, 10, 11
DASH_FIRST_ROW = DASH_HEADER_ROW + 2

# B to P is the P&L's own geometry, so the two sheets read the same way. Q is a gap and
# the three columns after it are the comparison.
DASH_LABEL_COLUMN = PL_LABEL_COLUMN
DASH_FIRST_MONTH_COLUMN = PL_FIRST_MONTH_COLUMN
DASH_SPACER_COLUMN = PL_SPACER_COLUMN
DASH_TOTAL_COLUMN = PL_TOTAL_COLUMN
DASH_GAP_COLUMN = DASH_TOTAL_COLUMN + 1
DASH_COMPARE_COLUMN, DASH_VARIANCE_COLUMN, DASH_PERCENT_COLUMN = (DASH_GAP_COLUMN + 1,
                                                                  DASH_GAP_COLUMN + 2,
                                                                  DASH_GAP_COLUMN + 3)
DASH_WIDTHS = [34] + [11] * len(PL_MONTHS) + [2, 12, 2, 13, 13, 10]

# The matrix goes under the block rather than beside it, so neither one has to be scrolled
# past to reach the other. Its columns are its own: C to F are the countries, H the total.
# G is left blank between them and not hidden - it is May in the month block above.
MATRIX_TITLE_ROW = DASH_FIRST_ROW + len(PL_ROWS) + 2
MATRIX_HEADER_ROW = MATRIX_TITLE_ROW + 2
MATRIX_FIRST_ROW = MATRIX_HEADER_ROW + 2
MATRIX_FIRST_COLUMN = 3
MATRIX_TOTAL_COLUMN = MATRIX_FIRST_COLUMN + len(COUNTRIES) + 1

# Where every line ends up in each of the two blocks, worked out by the same function the
# P&L sheets use - so nothing below has to know a row number.
DASH_ROW_AT = pl_row_numbers(PL_ROWS, DASH_FIRST_ROW)
MATRIX_ROW_AT = pl_row_numbers(PL_ROWS, MATRIX_FIRST_ROW)

COUNTRY_CELL, YEAR_CELL = f"$C${DASH_SELECT_ROW}", f"$E${DASH_SELECT_ROW}"
COMPARE_COUNTRY_CELL, COMPARE_YEAR_CELL = (f"$C${DASH_COMPARE_ROW}",
                                           f"$E${DASH_COMPARE_ROW}")


def pl_lookup(country_ref, year_ref, row, column):
    """One cell off a P&L sheet, picked by whatever the dropdowns say.

    `country_ref` may be a cell holding a country code or the word All, or a code written
    in as a literal - the matrix uses the second form, one column per country. All adds
    the four sheets up, which is only ever right for money; a ratio is worked out from its
    own lines instead.
    """
    address = f"ADDRESS({row},{column})"

    def off(sheet):
        return f"""INDIRECT("'{sheet}'!"&{address})"""

    one = off(f'{PL_SHEET_PREFIX} "&{country_ref}&" "&{year_ref}&"')
    if not country_ref.startswith("$"):
        return one                       # a literal country never means the group
    group = "+".join(off(f'{PL_SHEET_PREFIX} {label(country)} "&{year_ref}&"')
                     for country in COUNTRIES)
    return f'IF({country_ref}="{ALL_COUNTRIES}",{group},{one})'


def dash_ratio(letter, rule, rows):
    """One row over another, in whichever column it is asked for.

    Always worked out here rather than pulled, so a ratio is right under All as well as
    under one country - and so the comparison and total columns need no special case.
    """
    numerator, denominator = rule
    return f'=IFERROR({letter}{rows[numerator]}/{letter}{rows[denominator]},"")'


def write_dash_line(sheet, row, column, kind, formula, number_format):
    """One figure on a dashboard, in the house style for its kind of line."""
    cell = sheet.cell(row=row, column=column, value=formula)
    cell.font = Font(name=FACE, size=10, bold=kind == "total", color=rgb(INK))
    cell.number_format = number_format
    cell.alignment = Alignment(horizontal="right")
    if kind == "total":
        cell.border = TOP_RULE
    return cell


def write_dashboard(book):
    """A P&L pulled off whichever sheet the dropdowns name, and what it compares to."""
    sheet = new_sheet(book, DASHBOARD_SHEET, DASH_WIDTHS)
    write_title(sheet, DASHBOARD_SHEET)
    write_note(sheet, "View of all countries & years, uses the budget sheets as input. "
                      "Sheet is protected as its all calculation")

    # --- the selections
    countries = list(COUNTRY_LABELS.values()) + [ALL_COUNTRIES]
    for row, caption, country, year in ((DASH_SELECT_ROW, "Country", label(COUNTRIES[0]),
                                         BUDGET_YEARS[0]),
                                        (DASH_COMPARE_ROW, "Compare with", ALL_COUNTRIES,
                                         BUDGET_YEARS[0])):
        sheet.cell(row=row, column=2, value=caption).font = HEADER
        sheet.cell(row=row, column=4, value="Year").font = HEADER
        for column, value in ((3, country), (5, year)):
            cell = sheet.cell(row=row, column=column, value=value)
            cell.font = ENTRY                     # blue: these two are the ones you touch
            cell.alignment = Alignment(horizontal="center")
            # The only cells left open once the sheet is protected below.
            cell.protection = Protection(locked=False)
    add_dropdown(sheet, "C", countries, DASH_SELECT_ROW, DASH_COMPARE_ROW)
    add_dropdown(sheet, "E", BUDGET_YEARS, DASH_SELECT_ROW, DASH_COMPARE_ROW)

    # --- what the block is showing, and whether each month is closed or still a budget
    caption = sheet.cell(row=DASH_CAPTION_ROW, column=DASH_LABEL_COLUMN,
                         value=f'={COUNTRY_CELL}&" "&{YEAR_CELL}&"  -  {PL_TITLE}"')
    caption.font = YEAR_FONT
    for offset in range(len(PL_MONTHS)):
        column = DASH_FIRST_MONTH_COLUMN + offset
        # Pinned to Denmark: the basis is the same on every sheet for a given year, and
        # taking it off the selection would blank the row whenever All is chosen.
        cell = sheet.cell(row=DASH_BASIS_ROW, column=column, value="=" + pl_lookup(
            f'"{label(COUNTRIES[0])}"', YEAR_CELL, PL_BASIS_ROW, column))
        cell.font, cell.alignment = NOTE, Alignment(horizontal="center")

    # --- the headers
    headers = [(DASH_LABEL_COLUMN, PL_UNIT)] + [
        (DASH_FIRST_MONTH_COLUMN + offset, month) for offset, month in enumerate(PL_MONTHS)]
    headers += [(DASH_TOTAL_COLUMN, "Total"), (DASH_COMPARE_COLUMN, None),
                (DASH_VARIANCE_COLUMN, "Variance"), (DASH_PERCENT_COLUMN, "Var %")]
    for column, text in headers:
        cell = sheet.cell(row=DASH_HEADER_ROW, column=column, value=text)
        cell.font, cell.border = HEADER, UNDERLINE
        cell.alignment = Alignment(horizontal="left" if column == DASH_LABEL_COLUMN
                                   else "right")
    # The comparison column names itself, so it is obvious which way the variance runs.
    sheet.cell(row=DASH_HEADER_ROW, column=DASH_COMPARE_COLUMN,
               value=f'={COMPARE_COUNTRY_CELL}&" "&{COMPARE_YEAR_CELL}')

    # --- the lines
    write_dash_block(sheet, PL_ROWS, DASH_ROW_AT)
    for column in (DASH_SPACER_COLUMN, DASH_GAP_COLUMN):
        sheet.column_dimensions[get_column_letter(column)].hidden = True
    sheet.freeze_panes = f"{get_column_letter(DASH_FIRST_MONTH_COLUMN)}{DASH_FIRST_ROW}"
    write_dash_matrix(sheet)
    # Every figure here is a formula reading a P&L sheet, so there is nothing on it worth
    # typing over. Locked, bar the four dropdowns, which are the point of the sheet.
    sheet.protection.sheet = True
    return sheet


def write_dash_block(sheet, rows, rows_at):
    """The twelve months, the year, and the comparison beside it, line by line."""
    total_letter = get_column_letter(DASH_TOTAL_COLUMN)
    compare_letter = get_column_letter(DASH_COMPARE_COLUMN)
    variance_letter = get_column_letter(DASH_VARIANCE_COLUMN)

    for key, text, kind, reads in rows:
        if not key:
            continue                  # a blank or a gap holds the layout and nothing else
        row = rows_at[key]

        cell = sheet.cell(row=row, column=DASH_LABEL_COLUMN, value=text)
        cell.font = Font(name=FACE, size=10, bold=kind in ("title", "total"),
                         color=rgb(INK))
        if kind == "unused":
            sheet.row_dimensions[row].hidden = not PL_SHOW_UNUSED
            continue
        if kind == "title":
            continue

        ratio = kind == "ratio"
        money = PL_PERCENT if ratio else PL_MONEY
        for offset in range(len(PL_MONTHS)):
            column = DASH_FIRST_MONTH_COLUMN + offset
            letter = get_column_letter(column)
            formula = (dash_ratio(letter, reads, rows_at) if ratio
                       else "=" + pl_lookup(COUNTRY_CELL, YEAR_CELL, PL_ROW_AT[key], column))
            write_dash_line(sheet, row, column, kind, formula, money)

        # The year. Summed from the months for money, so it cannot disagree with them.
        first = get_column_letter(DASH_FIRST_MONTH_COLUMN)
        last = get_column_letter(DASH_FIRST_MONTH_COLUMN + len(PL_MONTHS) - 1)
        total = (dash_ratio(total_letter, reads, rows_at) if ratio
                 else f"=SUM({first}{row}:{last}{row})")
        write_dash_line(sheet, row, DASH_TOTAL_COLUMN, kind, total, money)

        compare = (dash_ratio(compare_letter, reads, rows_at) if ratio
                   else "=" + pl_lookup(COMPARE_COUNTRY_CELL, COMPARE_YEAR_CELL,
                                        PL_ROW_AT[key], PL_TOTAL_COLUMN))
        write_dash_line(sheet, row, DASH_COMPARE_COLUMN, kind, compare, money)
        write_dash_line(sheet, row, DASH_VARIANCE_COLUMN, kind,
                        f"={total_letter}{row}-{compare_letter}{row}", money)
        # Left off the ratios: a percentage change in a percentage reads as a margin and
        # is not one. The variance beside it is already in percentage points.
        if not ratio:
            write_dash_line(sheet, row, DASH_PERCENT_COLUMN, kind,
                            f'=IFERROR({variance_letter}{row}/ABS({compare_letter}{row}),"")',
                            PL_PERCENT)


def write_dash_matrix(sheet):
    """Every country in its own column for the selected year, and the group beside them."""
    title = sheet.cell(row=MATRIX_TITLE_ROW, column=DASH_LABEL_COLUMN,
                       value=f'="All countries  -  "&{YEAR_CELL}')
    title.font = YEAR_FONT

    columns = [(MATRIX_FIRST_COLUMN + offset, label(country))
               for offset, country in enumerate(COUNTRIES)]
    for column, text in ([(DASH_LABEL_COLUMN, PL_UNIT)] + columns
                         + [(MATRIX_TOTAL_COLUMN, "Total")]):
        cell = sheet.cell(row=MATRIX_HEADER_ROW, column=column, value=text)
        cell.font, cell.border = HEADER, UNDERLINE
        cell.alignment = Alignment(horizontal="left" if column == DASH_LABEL_COLUMN
                                   else "right")

    first = get_column_letter(MATRIX_FIRST_COLUMN)
    last = get_column_letter(MATRIX_FIRST_COLUMN + len(COUNTRIES) - 1)
    total_letter = get_column_letter(MATRIX_TOTAL_COLUMN)

    for key, text, kind, reads in PL_ROWS:
        if not key:
            continue
        row = MATRIX_ROW_AT[key]
        cell = sheet.cell(row=row, column=DASH_LABEL_COLUMN, value=text)
        cell.font = Font(name=FACE, size=10, bold=kind in ("title", "total"),
                         color=rgb(INK))
        if kind == "unused":
            sheet.row_dimensions[row].hidden = not PL_SHOW_UNUSED
            continue
        if kind == "title":
            continue

        ratio = kind == "ratio"
        money = PL_PERCENT if ratio else PL_MONEY
        for column, code in columns:
            letter = get_column_letter(column)
            formula = (dash_ratio(letter, reads, MATRIX_ROW_AT) if ratio
                       else "=" + pl_lookup(f'"{code}"', YEAR_CELL, PL_ROW_AT[key],
                                            PL_TOTAL_COLUMN))
            write_dash_line(sheet, row, column, kind, formula, money)
        group = (dash_ratio(total_letter, reads, MATRIX_ROW_AT) if ratio
                 else f"=SUM({first}{row}:{last}{row})")
        write_dash_line(sheet, row, MATRIX_TOTAL_COLUMN, kind, group, money)


### The three result tables

Each one goes out **long**, not wide: one row per observation, with every dimension in
its own column. A wide table - dates across the top, say - reads a little better on
screen and is worse at everything else, because reshaping it is a step that has to
happen before any join, pivot or filter can.

So the shape is the same on all three, and only the measure and the date change:

| | dated by | week from | rows |
|---|---|---|---|
| **Work day share** | invoice date | invoice date | stream x country x work day |
| **Due date cash** | due date | due date | stream x country x due date |
| **Payment date cash** | payment date | payment date | stream x country x payment date |

Each sheet's `Week` is the ISO week of *its own* date, so the three are bucketed by
three different clocks - invoiced, owed, paid - and comparing week 15 across the sheets
is what shows the delay moving the money.

The two cash tables are summed over payment terms. The term is what produces the spread
of dates, but once each row is dated the term itself is no longer a dimension anyone
reads the cash by - keeping it would multiply the row count by twelve to say the same
thing.

There is no `Cash share` column any more. It existed to hold the duty gross-up, and now
that duty is a stream of its own there is nothing for it to hold: every row is a plain
share of its own stream's budget.

### From shares to money

Every row carries an amount as well as a share, looked up from the budget sheet:

```
Cash amount = SUMIFS(BudgetInput[Amount], on Key and Stream) x Share of month
```

It is a **formula**, not a number the notebook worked out. That is the whole point of
having an input sheet: type a figure into `Budget input` and every row that depends on
it moves, without the notebook running again. The lookup uses structured references, so
it also grows by itself when rows are added to the budget table.

`Key` and `Stream` together pick exactly one budget row, which is what makes `SUMIFS`
safe as a lookup - it is summing a single cell. Put two rows in the budget for the same
key and stream, an actual and a forecast side by side, and it would quietly add them.

`Work day share` needed a `Stream` column to carry an amount at all. The share itself is
identical across all four streams - the same invoiced days drive each - so that column
repeats every share four times. The alternative was four amount columns, which is the
wide shape this export exists to avoid.

**The two amount columns are not the same measure.** `Invoiced amount` is what gets
billed; `Cash amount` is what moves, on the date it moves. Over the six months in scope
each stream's cash amounts total exactly its budget for those months - the arithmetic
check worth re-running whenever any of this changes.

**On currency.** Every budget is stated in DKK, whichever country it is for, so there is
one reporting currency in the workbook and a column of amounts is addable across
countries. The rate each country's budget was translated at is not held anywhere here -
it is applied before a figure is typed on a P&L, so nothing in this file can check it or
restate the budget if the rate moves.

### The charts beside them

**Four panels per sheet, one per country**, stacked down the right of the table starting
below the frozen rows, each beside the small `SUMIFS` block it is drawn from. Summing out
of the table in formulas rather than writing numbers from pandas is what keeps a chart
live if a row is ever filtered or edited.

One panel per country is not decoration. A percentage here is a share of *one* country's
budget, so four countries on one pair of axes is either four series fighting for the same
space or, worse, one series adding four things that cannot be added. Splitting them is
the same decision the charts in sections 4 and 10 make, and one currency throughout does
not change it: a share of Denmark's budget still means nothing beside a share of
Sweden's.

On the two cash sheets each panel draws **money in above the line and money out below
it**, with the service and the duty riding on it **stacked** in the same hue, the duty in
the lighter shade. The height of a stack is that side's total, which is the quickest way
to see how much of the flow is duty passing through rather than business being done. The
axis is formatted so the lower half reads positive.

The warning carries over: the four streams are shares of four different budgets, so their
shapes can be compared and their heights cannot - not until the amounts are applied.

**`Payment date cash` gets a second chart under each of those**, showing the net only:
money in less money out, one bar a week. It is the sheet that becomes the cash flow
model, and the net of what is merely *owed* is not a cash position, so `Due date cash`
does not get one.

The net chart is drawn from **`Cash amount`, not the shares**, and that is the whole
reason it can exist at all. A signed sum of the four shares would be adding fractions of
four different budgets together - the thing every other chart here is arranged to avoid.
In money the sum is real: duty collected genuinely does offset duty paid. So the net column runs its own `SUMIFS` over the money column
rather than adding up the four cells beside it.

It is also where the duty float shows itself. The weeks that go below zero are the ones
where COGS and the duty on it have been paid and the matching collection has not landed.

In [ ]:
def keyed(frame, date_column):
    """The export dimensions: year, month, country code, the key, and the week of the date.

    The week comes from whichever date the sheet is built around, so each sheet is
    bucketed by its own clock - invoiced, owed, or paid.
    """
    year, month = frame["period"].dt.year, frame["period"].dt.month
    country = frame["country"].map(COUNTRY_LABELS)
    weeks = frame[date_column].dt.isocalendar()
    return frame.assign(
        Year=year, Month=month, Country=country,
        # Zero-padded here only, so the keys sort in date order. The Month column stays 1-12.
        Key=year.astype(str) + "-" + month.astype(str).str.zfill(2) + "-" + country,
        Week=weeks.week.astype(int),
        # And the week with its year on the front, which is what anything grouping by
        # week has to use. Two things make a bare 1-53 ambiguous over a horizon this
        # long: it repeats every year, and an ISO week can start in one year and belong
        # to the next - so this takes the ISO year, not the invoice month's.
        **{"Week key": (weeks.year.astype(int).astype(str) + "-W"
                        + weeks.week.astype(int).astype(str).str.zfill(2))})


def in_column_order(frame, columns):
    """The frame cut down to the export columns, in the order the sheet writes them."""
    return frame[[header for header, _, _, _ in columns]]


def write_amounts(sheet, columns, row_count, amount_header):
    """Turn each row's share into money, by looking its budget up on the input sheet.

    A formula and not a number, because the point of having a budget sheet at all is
    that typing a figure into it moves everything downstream. Structured references
    (`BudgetInput[Amount]`) rather than a fixed range, so the lookup grows by itself
    when rows are added to that table.

    Key and Stream together pick exactly one budget row, which is what makes SUMIFS
    safe to use as a lookup here - it is summing a single cell. Put two rows in the
    budget for the same key and stream, an actual and a forecast side by side, and it
    would quietly add them instead.
    """
    key, stream = column_of(columns, "Key"), column_of(columns, "Stream")
    share = column_of(columns, "Share of month")
    amount = column_of(columns, amount_header)
    for row in range(FIRST_DATA_ROW, FIRST_DATA_ROW + row_count):
        # write_table already styled the cell; only the value is missing.
        sheet[f"{amount}{row}"] = (
            f"=SUMIFS({BUDGET_TABLE}[Amount],{BUDGET_TABLE}[Key],${key}{row},"
            f"{BUDGET_TABLE}[Stream],${stream}{row})*${share}{row}")


# --- what the charts will hold ---------------------------------------------------------
# Worked out here in pandas purely so the labels can be thinned: a bar too short to
# carry its own number is left bare, and that decision has to be made while the file is
# being written. The cells stay SUMIFS, so the charts themselves remain live.

def series_values(frame, criteria, measure, sign, weeks):
    """What one panel series will come to, week by week."""
    subset = frame
    for header, value in criteria.items():
        subset = subset[subset[header] == value]
    totals = subset.groupby("Week key")[measure].sum()
    return [float(totals.get(week, 0.0)) * sign for week in weeks]


def net_values(frame, code, weeks):
    """The net line for one country, in money - the only unit the legs net in.

    Signed off the flow rather than off the stream, which is the whole reason that column
    exists: the fee is money out on a stream whose other three legs are money in.
    """
    amounts = budget_lookup()
    subset = frame[frame["Country"] == code]
    signed = [amounts.get((key, stream), 0.0) * share * (1 if flow == "In" else -1)
              for key, stream, flow, share in zip(subset["Key"], subset["Stream"],
                                                  subset["Flow"], subset["Share of month"])]
    totals = (pd.DataFrame({"week": subset["Week key"].to_numpy(), "net": signed})
                .groupby("week")["net"].sum())
    return [float(totals.get(week, 0.0)) for week in weeks]


# --- 1. share of the month per work day ----------------------------------------------
SHARE_COLUMNS = [
    ("Key",             18, "@",          "left"),
    ("Year",             8, "0",          "center"),
    ("Month",            8, "0",          "center"),
    ("Country",         10, "@",          "center"),
    ("Stream",          14, "@",          "left"),
    ("Invoice date",    14, "yyyy-mm-dd", "center"),
    ("Week",             8, "0",          "center"),
    ("Week key",        11, "@",          "center"),
    ("Share of month",  15, "0.000%",     "right"),
    ("Invoiced amount", 16, "#,##0",      "right"),
]


def write_work_day_share(book):
    """Schema 1: what each work day is worth, as a share of the month it sits in."""
    # The share itself does not depend on the stream - the same invoiced frame drives
    # all four - but the amount does, so the stream has to be a column here too. That
    # keeps all three sheets the same shape at the cost of repeating each share.
    frame = pd.concat([keyed(invoiced, "invoice_date").assign(Stream=name)
                       for name in STREAM_LABELS.values()], ignore_index=True)
    frame = frame.rename(columns={"invoice_date": "Invoice date",
                                  "pct_of_month": "Share of month"})
    frame["Invoiced amount"] = None        # filled in below, as a formula
    frame = in_column_order(frame, SHARE_COLUMNS).sort_values(["Stream", "Country",
                                                               "Invoice date"])

    sheet = new_sheet(book, "Work day share", [width for _, width, _, _ in SHARE_COLUMNS])
    write_title(sheet, "Work day share")
    rows = write_table(sheet, "WorkDayShare", SHARE_COLUMNS, frame.itertuples(index=False))
    write_amounts(sheet, SHARE_COLUMNS, rows, "Invoiced amount")

    # Pinned to one stream: all four carry identical shares here, so a panel that took
    # them all would count every work day four times.
    weeks = sorted(frame["Week key"].unique())
    panels = []
    for code, colour in zip(COUNTRY_LABELS.values(), PALETTE):
        criteria = {"Country": code, "Stream": STREAM_LABELS["revenue"]}
        panels.append((code, [(code, criteria, colour, 1,
                               series_values(frame, criteria, "Share of month", 1, weeks))],
                       None))
    add_week_panels(sheet, SHARE_COLUMNS, rows, weeks, panels,
                    measure="Share of month", chart_title="invoicing by ISO week",
                    y_title="% of that month")
    return sheet


# --- 2 and 3. cash, dated two different ways ------------------------------------------
def cash_line(stream, leg):
    """What a row is called on the cash flow sheets.

    The fee is the one that is not simply its stream. It comes off revenue, but it is the
    bank's, so it gets a line of its own instead of netting into the revenue above it.
    """
    return LEG_LABELS["fee"] if leg == "fee" else STREAM_LABELS[stream]


def cash_columns(date_header):
    """The cash schema. Only the name of the date column separates the two sheets.

    Three columns carry the factoring: the `Leg` a row settles on, the `Flow` that puts
    it above or below the line, and the `Leg rate` the workbook applies. `Invoiced share`
    is what was billed and `Share of month` what survives the leg - the second is the
    first times the rate, and everything downstream reads the second.
    """
    return [
        ("Key",            18, "@",          "left"),
        ("Year",            8, "0",          "center"),
        ("Month",           8, "0",          "center"),
        ("Country",        10, "@",          "center"),
        ("Stream",         14, "@",          "left"),
        ("Leg",            19, "@",          "left"),
        ("Flow",            7, "@",          "center"),
        ("Cash line",      19, "@",          "left"),
        (date_header,      14, "yyyy-mm-dd", "center"),
        ("Week",            8, "0",          "center"),
        ("Week key",       11, "@",          "center"),
        ("Cash week",      14, "@",          "center"),
        ("Cash key",       38, "@",          "left"),
        ("Invoiced share", 15, "0.000%",     "right"),
        ("Leg rate",       11, "0.00%",      "right"),
        ("Share of month", 15, "0.000%",     "right"),
        ("Cash amount",    16, "#,##0",      "right"),
    ]


def write_rates(sheet, columns, row_count):
    """The rate a row's leg carries, and what is left of its share once it is applied.

    Both formulas, so the advance and the fee can be argued about in the workbook rather
    than here. The rate is looked up on one composite key: the row's own `Key`, which is
    already the month and country it was invoiced in, with its stream and leg on the end.
    That is the same key the rate table builds for itself out of the same four things, so
    a lookup always finds its row - and it is one SUMIFS condition rather than four, on
    sheets that ask the question tens of thousands of times over.

    The month is in there because the facility is set per month now. A rate looked up on
    country alone would have gone on answering, with the wrong month's figures.
    """
    key, stream = column_of(columns, "Key"), column_of(columns, "Stream")
    leg, invoiced = column_of(columns, "Leg"), column_of(columns, "Invoiced share")
    rate, share = column_of(columns, "Leg rate"), column_of(columns, "Share of month")
    for row in range(FIRST_DATA_ROW, FIRST_DATA_ROW + row_count):
        sheet[f"{rate}{row}"] = (
            f'=SUMIFS({RATE_TABLE}[Rate],{RATE_TABLE}[Rate key],'
            f'${key}{row}&"|"&${stream}{row}&"|"&${leg}{row})')
        sheet[f"{share}{row}"] = f"=${invoiced}{row}*${rate}{row}"


def country_panels(frame, weeks, with_net):
    """One panel per country: the service, the duty riding on it, the bank's cut, the net.

    Each stream series is pinned to its own side of the flow, and that is what keeps the
    fee out of the revenue it is charged on: a fee row carries the same stream as the
    revenue it came off and the opposite flow, so naming the side excludes it.
    """
    order = list(STREAMS_IN) + list(STREAMS_OUT)
    panels = []
    for code in COUNTRY_LABELS.values():
        series = []
        for stream in order:
            side = "In" if stream in STREAMS_IN else "Out"
            criteria = {"Country": code, "Stream": STREAM_LABELS[stream], "Flow": side}
            sign = 1 if side == "In" else -1
            series.append((STREAM_LABELS[stream], criteria, STREAM_COLOURS[stream], sign,
                           series_values(frame, criteria, "Share of month", sign, weeks)))
        # Both streams' fees on one line, because the bank takes them as one deduction.
        criteria = {"Country": code, "Leg": LEG_LABELS["fee"]}
        series.append((LEG_LABELS["fee"], criteria, PALETTE[3], -1,
                       series_values(frame, criteria, "Share of month", -1, weeks)))
        panels.append((code, series, net_values(frame, code, weeks) if with_net else None))
    return panels


def write_cash(book, name, table_name, date_column, date_header, chart_title, net=None):
    """Schemas 2 and 3: the same cash table, dated by when it is owed or by when it lands."""
    columns = cash_columns(date_header)
    # Summed over terms: the term set the date, and past that it is not a dimension
    # anyone reads the cash by. The leg is one, so it stays.
    totals = (payments.groupby(["stream", "leg", "country", "period", date_column],
                               as_index=False)["pct_of_month"].sum())
    frame = keyed(totals, date_column).rename(columns={date_column: date_header,
                                                       "pct_of_month": "Invoiced share"})
    frame["Stream"] = frame["stream"].map(STREAM_LABELS)
    frame["Leg"] = frame["leg"].map(LEG_LABELS)
    frame["Flow"] = [leg_flow(stream, leg)
                     for stream, leg in zip(frame["stream"], frame["leg"])]
    # What the cash flow sheets call this row, and the single key they find it on. One
    # criterion rather than three is worth a column here: these sheets are looked up
    # fifteen thousand times over and the table they read runs to tens of thousands of rows.
    frame["Cash line"] = [cash_line(stream, leg)
                          for stream, leg in zip(frame["stream"], frame["leg"])]
    frame["Cash week"] = frame[date_header].map(cash_week_of)
    frame["Cash key"] = frame["Country"] + "|" + frame["Cash line"] + "|" + frame["Cash week"]
    # Worked out here as well as on the sheet, for the same reason the amounts are: the
    # chart labels have to be thinned while the file is being written. Both cells are
    # overwritten with formulas below, so the workbook itself stays live.
    frame["Leg rate"] = [leg_rates(country, period, stream)[leg]
                         for country, period, stream, leg
                         in zip(frame["country"], frame["period"], frame["stream"],
                                frame["leg"])]
    frame["Share of month"] = frame["Invoiced share"] * frame["Leg rate"]
    frame["Cash amount"] = None            # filled in below, as a formula
    frame = in_column_order(frame, columns).sort_values(["Stream", "Leg", "Country",
                                                         date_header])

    sheet = new_sheet(book, name, [width for _, width, _, _ in columns])
    write_title(sheet, name)
    rows = write_table(sheet, table_name, columns, frame.itertuples(index=False))
    write_rates(sheet, columns, rows)
    write_amounts(sheet, columns, rows, "Cash amount")

    weeks = sorted(frame["Week key"].unique())
    add_week_panels(sheet, columns, rows, weeks,
                    country_panels(frame, weeks, with_net=net is not None),
                    measure="Share of month", chart_title=chart_title,
                    y_title="% of that stream's budget", net=net)
    return sheet


## The cash flow sheets

One per country per year, laid out the way the P&L sheets are so the two read together,
except that the columns are weeks and there are three years rather than two. 2028 is there
because December 2027's invoicing on sixty-day terms is 2028's cash.

Each month's weeks sit under it and that month's own total sits at the end of them. The
week columns are grouped, so the buttons above them collapse a month down to its total and
the sheet becomes a monthly view without needing a second tab. A week straddling two months
is a column under each, holding only that month's days, so a month total is a real calendar
month and ties to the budget month behind it.

The rows are the two families the model already has. Revenue, its duty, COGS and its duty
come off the payment date sheet, and the twelve scheduled lines off the cost cash sheet.
The factoring fee gets a line of its own rather than netting into revenue, because it is
the bank's money and not a smaller sale.

Every figure is one SUMIFS on a single composite key - country, line and week joined into
one string on both source tables. Three criteria would have been the obvious way to write
it and about three times the work, on a lookup that happens fifteen thousand times over a
table of tens of thousands of rows.

Everything is shown in thousands, like the P&L, by number format rather than by dividing.
The cell still holds whole units, so it ties back to the payment date sheet if you click
it.

In [ ]:
CF_SHEET_PREFIX = "CF"
CF_TITLE = "Cash flow, week by week"
PAYMENT_TABLE = "PaymentDateCash"   # the name write_cash() is asked for in the build cell

# The years cash lands in, taken from the payments themselves rather than from the budget:
# the tail of a horizon always runs past the last month anybody budgeted.
CASH_YEARS = tuple(range(int(payments["payment_date"].min().year),
                         int(payments["payment_date"].max().year) + 1))

CF_YEAR_ROW, CF_MONTH_ROW, CF_HEADER_ROW, CF_KEY_ROW = 4, 5, 6, 7
CF_FIRST_ROW = CF_KEY_ROW + 2
CF_LABEL_COLUMN, CF_LINE_COLUMN, CF_FIRST_DATA_COLUMN = 2, 3, 4

CF_LABEL_WIDTH, CF_WEEK_WIDTH, CF_TOTAL_WIDTH = 30, 9, 11
# The comma divides by a thousand for display only, so the cell still holds whole units and
# still ties to the table it was summed out of.
CF_MONEY = '#,##0,;(#,##0,);"-"'


def cash_rows():
    """The cash flow lines, built off the two families the model already has.

    Generated rather than typed out, so a cost line added in section 11 turns up here by
    itself instead of being quietly missing from the one sheet anybody reads.
    """
    rows = [("cash_in", "Cash in", "title", None),
            ("in_revenue", STREAM_LABELS["revenue"], "traded", STREAM_LABELS["revenue"]),
            ("in_duty", STREAM_LABELS["revenue_duty"], "traded",
             STREAM_LABELS["revenue_duty"]),
            ("in_total", "Cash in total", "total", ("in_revenue", "in_duty")),
            (None, "", "blank", None),
            ("cash_out", "Cash out", "title", None),
            ("out_cogs", STREAM_LABELS["cogs"], "traded", STREAM_LABELS["cogs"]),
            ("out_cogs_duty", STREAM_LABELS["cogs_duty"], "traded",
             STREAM_LABELS["cogs_duty"]),
            ("out_fee", LEG_LABELS["fee"], "traded", LEG_LABELS["fee"])]
    out = ["out_cogs", "out_cogs_duty", "out_fee"]

    for title, prefix in (("Manpower", "mp_"), ("SG&A", "sga_")):
        rows += [(None, "", "blank", None), (f"head_{prefix}", title, "title", None)]
        for key, name in SCHEDULED_LINES.items():
            if key.startswith(prefix):
                rows.append((f"out_{key}", name, "cost", name))
                out.append(f"out_{key}")

    rows += [(None, "", "blank", None),
             ("out_total", "Cash out total", "total", tuple(out)),
             (None, "", "blank", None),
             ("net", "Net cash flow", "net", ("in_total", "out_total"))]
    return rows


CASH_ROWS = cash_rows()
CASH_ROW_AT = pl_row_numbers(CASH_ROWS, CF_FIRST_ROW)


def cash_year_weeks(year):
    """Every (month, week column) a year needs, in order.

    A week straddling two months turns up under each of them. That is what keeps a month
    column a calendar month instead of a run of whole weeks - and it matters most exactly
    where it is most likely to: a payroll on the last working day of the month.
    """
    columns = []
    for month in range(1, 13):
        first = pd.Timestamp(year=year, month=month, day=1)
        weeks = []
        for day in pd.date_range(first, first + pd.offsets.MonthEnd(0)):
            key = cash_week_of(day)
            if key not in weeks:
                weeks.append(key)
        columns.append((month, weeks))
    return columns


def cf_sheet_name(country, year):
    """`CF DNK 2026`, beside `P&L DNK 2026`."""
    return f"{CF_SHEET_PREFIX} {label(country)} {year}"


def write_cash_flow(book, country, year):
    """One country's cash for one year: its weeks across, grouped under their months."""
    months = cash_year_weeks(year)
    code = label(country)

    # Work the geometry out first: which column each week lands in, and which column each
    # month's total lands in, so the formulas below never have to count.
    weeks, totals, column = {}, {}, CF_FIRST_DATA_COLUMN
    for month, keys in months:
        weeks[month] = []
        for key in keys:
            weeks[month].append((column, key))
            column += 1
        totals[month] = column
        column += 1
    year_column = column

    widths = ([CF_LABEL_WIDTH, 2]
              + [CF_WEEK_WIDTH] * (year_column - CF_FIRST_DATA_COLUMN) + [CF_TOTAL_WIDTH])
    sheet = new_sheet(book, cf_sheet_name(country, year), widths)
    write_title(sheet, f"{CF_TITLE} - {code} {year}")
    sheet.column_dimensions[get_column_letter(CF_LINE_COLUMN)].hidden = True

    # --- the three header tiers. Merged first and written after, the way the P&L does it:
    # a merge keeps the top-left cell and throws the rest away.
    sheet.merge_cells(start_row=CF_YEAR_ROW, start_column=CF_FIRST_DATA_COLUMN,
                      end_row=CF_YEAR_ROW, end_column=year_column)
    stamp = sheet.cell(row=CF_YEAR_ROW, column=CF_FIRST_DATA_COLUMN, value=year)
    stamp.font, stamp.number_format = YEAR_FONT, "0"
    stamp.alignment = Alignment(horizontal="center")
    sheet.cell(row=CF_HEADER_ROW, column=CF_LABEL_COLUMN, value=PL_UNIT).font = HEADER

    for month, keys in months:
        first, last = weeks[month][0][0], totals[month]
        sheet.merge_cells(start_row=CF_MONTH_ROW, start_column=first,
                          end_row=CF_MONTH_ROW, end_column=last)
        cell = sheet.cell(row=CF_MONTH_ROW, column=first, value=PL_MONTHS[month - 1])
        cell.font, cell.alignment = HEADER, Alignment(horizontal="center")
        for position, key in weeks[month]:
            head = sheet.cell(row=CF_HEADER_ROW, column=position, value=key.split("-")[-1])
            head.font, head.border = HEADER, UNDERLINE
            head.alignment = Alignment(horizontal="center")
            # The key the formulas below join on, kept out of sight under the label.
            sheet.cell(row=CF_KEY_ROW, column=position, value=key)
        head = sheet.cell(row=CF_HEADER_ROW, column=last, value="Total")
        head.font, head.border = HEADER, UNDERLINE
        head.alignment = Alignment(horizontal="center")
        # Grouped so the month collapses to its total, which is the monthly view.
        sheet.column_dimensions.group(get_column_letter(weeks[month][0][0]),
                                      get_column_letter(weeks[month][-1][0]),
                                      outline_level=1, hidden=False)
    head = sheet.cell(row=CF_HEADER_ROW, column=year_column, value="Year")
    head.font, head.border = HEADER, UNDERLINE
    head.alignment = Alignment(horizontal="center")
    sheet.row_dimensions[CF_KEY_ROW].hidden = True
    sheet.sheet_properties.outlinePr.summaryRight = True
    # Grouping a range replaces the widths inside it, so the totals are sized after it.
    for month, _ in months:
        sheet.column_dimensions[get_column_letter(totals[month])].width = CF_TOTAL_WIDTH

    # --- the lines
    month_letters = [get_column_letter(totals[month]) for month, _ in months]
    for key, text, kind, reads in CASH_ROWS:
        if not key:
            continue
        row = CASH_ROW_AT[key]
        cell = sheet.cell(row=row, column=CF_LABEL_COLUMN, value=text)
        cell.font = Font(name=FACE, size=10, bold=kind in ("title", "total", "net"),
                         color=rgb(INK))
        if kind == "title":
            continue
        if kind in ("traded", "cost"):
            sheet.cell(row=row, column=CF_LINE_COLUMN, value=reads)

        for month, _ in months:
            for position, _ in weeks[month]:
                letter = get_column_letter(position)
                if kind == "traded":
                    formula = (f'=SUMIFS({PAYMENT_TABLE}[Cash amount],'
                               f'{PAYMENT_TABLE}[Cash key],"{code}|"&$'
                               f'{get_column_letter(CF_LINE_COLUMN)}{row}&"|"&'
                               f'{letter}${CF_KEY_ROW})')
                elif kind == "cost":
                    formula = (f'=SUMIFS({COST_CASH_TABLE}[Cash amount],'
                               f'{COST_CASH_TABLE}[Cash key],"{code}|"&$'
                               f'{get_column_letter(CF_LINE_COLUMN)}{row}&"|"&'
                               f'{letter}${CF_KEY_ROW})')
                elif kind == "net":
                    formula = (f"={letter}{CASH_ROW_AT[reads[0]]}"
                               f"-{letter}{CASH_ROW_AT[reads[1]]}")
                else:
                    formula = "=" + "+".join(f"{letter}{CASH_ROW_AT[member]}"
                                             for member in reads)
                write_cf_cell(sheet, row, position, kind, formula)

        # The month, then the year off the months - never off the weeks, which a
        # straddling week would have counted into two months at once.
        for month, _ in months:
            first = get_column_letter(weeks[month][0][0])
            last = get_column_letter(weeks[month][-1][0])
            write_cf_cell(sheet, row, totals[month], kind,
                          f"=SUM({first}{row}:{last}{row})")
        write_cf_cell(sheet, row, year_column, kind,
                      "=" + "+".join(f"{letter}{row}" for letter in month_letters))

    sheet.freeze_panes = f"{get_column_letter(CF_FIRST_DATA_COLUMN)}{CF_FIRST_ROW}"
    return sheet


def write_cf_cell(sheet, row, column, kind, formula):
    """One figure on a cash flow sheet, in the house style for its kind of line."""
    cell = sheet.cell(row=row, column=column, value=formula)
    cell.font = Font(name=FACE, size=10, bold=kind in ("total", "net"), color=rgb(INK))
    cell.number_format = CF_MONEY
    cell.alignment = Alignment(horizontal="right")
    if kind in ("total", "net"):
        cell.border = TOP_RULE
    return cell


### How the workbook is laid out

Four groups, in the order the numbers move through them:

| group | sheets | what they are |
|---|---|---|
| **Budget 2026** | `P&L DNK 2026` ... `P&L USA 2026` | where the budget is actually typed |
| **Budget 2027** | `P&L DNK 2027` ... `P&L USA 2027` | the same, a year on |
| **Inputs** | `Model assumptions`, `Budget input` | what the model assumes, and the budget in long form |
| **Python Data** | `Work day share`, `Due date cash`, `Payment date cash` | written by this notebook, and overwritten every time it runs |

The budget comes **first**, because it is the only group anybody edits. Everything to the
right of it is downstream: `Budget input` reads the P&L sheets, and the three data sheets
read `Budget input`. Left to right is both the order the sheets are read and the order they
are calculated.

A divider tab stands in front of each group - `Budget 2026 -->`, `Budget 2027 -->`,
`Inputs -->`, `Python Data -->` - holding nothing at all. It is the only kind of tab with a
colour, so the grouping shows in the tab strip at the bottom of the window without a single
sheet having to be renamed to carry a prefix. The two budget dividers share a colour,
because they are the same kind of thing - the year in the name is what separates them.

The sheets are written in the order they are meant to be read, so nothing is moved
afterwards: `new_sheet()` appends, and the tab strip comes out in build order.

**Only two of the sheets are anybody's to edit, and neither of them is in `Inputs`.** The
eight P&Ls are. `Model assumptions` is a record of what the notebook did, `Budget input` is
a formula reading the P&Ls, and the three data sheets are output. Everything but the budget
is rewritten from scratch on the next run.

**Which is also the thing still wrong with it.** Re-running rewrites the eight P&L sheets
too, so a budget typed into them is lost. Still the outstanding item on the list below.

### The contents sheet

Thirty-odd tabs is more than the tab strip can show, so the first sheet in the workbook is
a list of the rest. It is built from the workbook itself rather than from a list kept
alongside it - the groups are read off the divider tabs that were already there, so a sheet
added later turns up on it without anybody remembering to add it.

Every other sheet gets a link back in its top-left corner, on the thin row above the title
that was margin until now. That row is a little taller than it was as a result, which is
the whole cost of never landing on a tab with no way back.

In [ ]:
CONTENTS_SHEET = "Contents"
SEPARATOR_MARK = "-->"          # what a divider tab's name ends with

LINK = Font(name=FACE, size=10, color=rgb(PALETTE[0]), underline="single")
BACK = Font(name=FACE, size=8, color=rgb(MUTED), underline="single")
BACK_ROW_HEIGHT = 12            # the top margin, given something to hold


def sheet_link(target, text):
    """A link to another sheet in this workbook.

    HYPERLINK rather than a stored relationship: it is one formula, it survives a sheet
    being rewritten, and it is the form anybody reading the cell will recognise.
    """
    # A sheet name goes inside single quotes so spaces and the & in "P&L" survive it.
    quote = "'"
    return f'=HYPERLINK("#{quote}{target}{quote}!A1","{text}")'


def sheet_groups(book):
    """The tabs in build order, cut into the groups the divider tabs already mark."""
    groups = []
    for name in book.sheetnames:
        if name.endswith(SEPARATOR_MARK):
            groups.append((name[:-len(SEPARATOR_MARK)].strip(), []))
        elif groups:
            groups[-1][1].append(name)
    return groups


def write_contents(book):
    """The front page: every sheet in the workbook, grouped and linked."""
    groups = sheet_groups(book)
    widths = [26, 2] * len(groups)
    sheet = new_sheet(book, CONTENTS_SHEET, widths[:-1])
    write_title(sheet, CONTENTS_SHEET)
    write_note(sheet, "Every sheet in the workbook. Each one links back here from the "
                      "row above its title.")

    # Groups across rather than down: the longest is a dozen tabs, and side by side they
    # all fit on one screen where one column of thirty would not.
    for index, (name, tabs) in enumerate(groups):
        column = 2 + index * 2
        head = sheet.cell(row=HEADER_ROW, column=column, value=name)
        head.font, head.border = HEADER, UNDERLINE
        for offset, tab in enumerate(tabs):
            cell = sheet.cell(row=FIRST_DATA_ROW + offset, column=column,
                              value=sheet_link(tab, tab))
            cell.font = LINK

    sheet.freeze_panes = f"B{FIRST_DATA_ROW}"
    book.move_sheet(CONTENTS_SHEET, offset=-(len(book.sheetnames) - 1))
    return sheet


def add_back_links(book):
    """A link home on every other sheet, so no tab is somewhere you get stuck."""
    for name in book.sheetnames:
        if name == CONTENTS_SHEET:
            continue
        sheet = book[name]
        # Row 1 was margin and nothing else. It is a little taller now and carries this.
        sheet.row_dimensions[1].height = BACK_ROW_HEIGHT
        cell = sheet.cell(row=1, column=2,
                          value=sheet_link(CONTENTS_SHEET, "< Contents"))
        cell.font = BACK


In [ ]:
def write_separator(book, name, colour):
    """A tab that holds nothing and only says where one group of sheets begins.

    The colour is the whole device: no other tab has one, so the dividers cut the strip
    into groups without any sheet needing a prefix on its name.
    """
    sheet = new_sheet(book, name, [])
    sheet.sheet_properties.tabColor = rgb(colour)
    write_title(sheet, name)
    return sheet


# --- build ----------------------------------------------------------------------------
book = Workbook()
book.remove(book.active)   # the sheet a new workbook comes with; ours all come from new_sheet()

# In the order they are read, which is also the order they are calculated: the budget is
# typed on the P&L sheets, the budget sheet reads those, and the data sheets read that.
# new_sheet() appends, so build order is tab order.
#
# A divider per year rather than one over all eight: eight tabs in a row is a wall, and a
# budget is worked through one year at a time. The countries sit inside a year in scope
# order, which is the order a budget is gone through rather than the order one country's
# story reads in.
for year in BUDGET_YEARS:
    write_separator(book, f"Budget {year} -->", PALETTE[2])
    for country in COUNTRIES:
        write_profit_and_loss(book, country, year)

write_separator(book, "Cash flow -->", PALETTE[3])
for year in CASH_YEARS:
    for country in COUNTRIES:
        write_cash_flow(book, country, year)

write_separator(book, "Inputs -->", PALETTE[0])
write_model_assumptions(book)
write_budget_input(book)
write_factoring(book)
write_cost_timing(book)
write_dashboard(book)

write_separator(book, "Python Data -->", PALETTE[1])
write_work_day_share(book)
write_cash(book, "Due date cash", "DueDateCash", "due_date", "Due date",
           "cash owed by ISO week")
# Only the payment-date sheet gets a net chart. It is the one that becomes the cash
# flow model, and the net of what is *owed* is not a cash position.
write_cost_cash(book)
write_cash(book, "Payment date cash", PAYMENT_TABLE, "payment_date", "Payment date",
           "cash landing by ISO week",
           net=("Cash amount", "net cash by ISO week", "net cash, DKK"))

# Last, so it can read the tab strip it is describing.
write_contents(book)
add_back_links(book)

# openpyxl writes formulas with no cached result, so Excel would open on columns of
# blanks - the keys and every amount among them. This tells it to recalculate on the
# way in.
book.calculation.fullCalcOnLoad = True

book.save(WORKBOOK)
[sheet.title for sheet in book]

## 15. Next

`payments` is the full grid: stream, country, invoice month, invoice date, term, due
date, expected date, payment date and share of month. The amounts now live in the
workbook rather than here, so applying them is a lookup in Excel rather than a join in
pandas. From here:

1. Replace the placeholder term mixes with the real ones.
2. Replace the equal work-day split with a real invoicing profile.
3. Check the generated bank holidays against a real source, per country. They are rules
   now and cover the horizon, but they are still best-effort ones.
4. Replace the random figures on the eight P&L sheets with the real budget. Everything
   downstream follows from there - `Budget input` reads the P&L, and the three data
   sheets read `Budget input`.
5. Decide where the FX rates live. Every P&L is in DKK, so the workbook needs no rate
   table to add its amounts up - but the rate that put a Norwegian budget into DKK is
   applied before anything is typed, and is not written down anywhere in here.

Two things this notebook still overwrites that it should not: re-running it rewrites the
eight P&L sheets, so any budget typed into them is lost. And the metrics log is still to
come.